# **Which degree programmes are aligned with the wider range of jobs in today's evolving labour market?**

This notebook studies broad programme-job alignment across a shared labour-market job pool. It does not evaluate intended-industry fit or standalone skill demand. Modules are compared against all retained jobs, threshold-free module features are built first, Spearman-informed module importance is used during aggregation, and the final threshold is calibrated only after the programme-job matrices are constructed.

The notebook is organized as one analytical story:
- Part A prepares the curriculum and labour-market data.
- Part B builds module and programme similarity objects.
- Part C calibrates the final decision rule.
- Part D reports semantic alignment, industry breadth, programme robustness, and limitations.


# Part A — Setup and Data


## Step 0: Setup and File Paths

**Purpose:** define the notebook environment, shared helpers, file paths, and analysis constants before any data is loaded.

**What is done:** imports, plotting configuration, programme-label helpers, project-root detection, file paths, `TOP_K`, `THRESHOLD_GRID`, `NULL_N_ITER`, and `SELECTED_SIMILARITY_THRESHOLD = None`.

**Why this comes first:** the downstream cells reuse these helpers and constants repeatedly, so setting them once at the top keeps the notebook reproducible and easy to follow.


In [ ]:
import ast
import json
import os
import re
import warnings
from collections import deque
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, MaxNLocator, PercentFormatter


plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["font.family"] = "DejaVu Sans"

school_colors = {
    "nus": {"primary": "#EF7C00", "secondary": "#003D7C"},
    "smu": {"primary": "#8A704C", "secondary": "#141C52"},
    "sutd": {"primary": "#A01735", "secondary": "#000000"},
}


def prettify_course_name(text):
    text = str(text).replace("_", " ").strip()
    return " ".join(word.capitalize() for word in text.split())


def build_programme_label(df, school_col="school", course_col="course"):
    out = df.copy()
    school_series = (
        out[school_col]
        .fillna("")
        .astype(str)
        .str.upper()
        .str.strip()
    )
    course_series = (
        out[course_col]
        .fillna("")
        .astype(str)
        .map(prettify_course_name)
        .str.strip()
    )
    out["programme_label"] = school_series + " | " + course_series
    out["programme_only_label"] = course_series
    return out


def add_programme_display_label(df, school_col="school", course_col="course"):
    out = build_programme_label(df, school_col=school_col, course_col=course_col)
    duplicate_names = set(
        out["programme_only_label"].value_counts().loc[lambda s: s > 1].index
    )
    out["programme_display_label"] = out["programme_only_label"]
    dup_mask = out["programme_only_label"].isin(duplicate_names)
    out.loc[dup_mask, "programme_display_label"] = (
        out.loc[dup_mask, "programme_only_label"]
        + " ("
        + out.loc[dup_mask, school_col].astype(str).str.upper()
        + ")"
    )
    return out


def minmax_normalize_columns(df, cols):
    out = df.copy()
    for col in cols:
        values = pd.to_numeric(out[col], errors="coerce")
        col_min = values.min()
        col_max = values.max()
        if pd.isna(col_min) or pd.isna(col_max):
            out[col] = np.nan
        elif np.isclose(col_min, col_max):
            out[col] = 0.5
        else:
            out[col] = (values - col_min) / (col_max - col_min)
    return out


def add_percentile_rank_overview_score(df, metric_cols, new_col="semantic_overview_score"):
    out = df.copy()
    missing_cols = [col for col in metric_cols if col not in out.columns]
    if missing_cols:
        raise KeyError(f"Missing required column(s): {missing_cols}")

    rank_df = pd.DataFrame(index=out.index)
    for col in metric_cols:
        rank_df[col] = pd.to_numeric(out[col], errors="coerce").rank(pct=True, method="average")

    out[new_col] = rank_df.mean(axis=1)
    return out


In [ ]:
def find_project_root(start: Path, markers=("README.md", ".git")):
    current = start.resolve()
    for parent in [current] + list(current.parents):
        if any((parent / marker).exists() for marker in markers):
            return parent
    return start.resolve()


START_DIR = Path.cwd()
PROJECT_ROOT = find_project_root(START_DIR)
BASE_DATA_PATH = PROJECT_ROOT / "DSA4264 Data"
COURSES_DIR = BASE_DATA_PATH / "NUS-SMU-SUTD Courses"

SCHOOLS = ["nus", "smu", "sutd"]
SCHOOL_DISPLAY = {"nus": "NUS", "smu": "SMU", "sutd": "SUTD"}

nus_path = COURSES_DIR / "10_nus_modules_embedded.parquet"
smu_path = COURSES_DIR / "10_smu_modules_embedded.parquet"
sutd_path = COURSES_DIR / "10_sutd_modules_embedded.parquet"
jobs_path = BASE_DATA_PATH / "processed_jobs_dual_embeddings.parquet"
spearman_path = BASE_DATA_PATH / "spearman_correlations_with_employment.csv"
ols_weights_path = BASE_DATA_PATH / "ols_learned_weights_by_university.csv"

MODULE_PARQUET_PATHS = {
    "nus": nus_path,
    "smu": smu_path,
    "sutd": sutd_path,
}
CURRICULUM_CSV_PATHS = {
    "nus": COURSES_DIR / "nus_courses.csv",
    "smu": COURSES_DIR / "smu_courses.csv",
    "sutd": COURSES_DIR / "sutd_courses.csv",
}
PREREQ_GRAPH_PATHS = {
    "nus": BASE_DATA_PATH / "nus_prerequisite_graph.json",
    "smu": None,
    "sutd": None,
}

TOP_K = 100
THRESHOLD_GRID = np.round(np.arange(0.20, 0.5001, 0.025), 3)
NULL_N_ITER = int(os.getenv("EMPLOYABILITY_NULL_N_ITER", "30"))
SELECTED_SIMILARITY_THRESHOLD = None

for path in [jobs_path, spearman_path, ols_weights_path, *MODULE_PARQUET_PATHS.values(), *CURRICULUM_CSV_PATHS.values()]:
    if path is not None and not Path(path).exists():
        raise FileNotFoundError(f"Missing required input: {path}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Jobs parquet: {jobs_path}")
print(f"Spearman table: {spearman_path}")
print(f"Null iterations for this run: {NULL_N_ITER}")


In [ ]:
jobs_df = pd.read_parquet(jobs_path)
jobs_df.head()

## Step 1: Data Loading and Preprocessing

**Purpose:** prepare the curriculum-side inputs that define programme structure.

**What is done:** load the embedded module parquets for NUS, SMU, and SUTD; load curriculum CSVs; merge curriculum weights back into the embedded module tables; standardize keys such as `course`, `type`, and `code`; and preserve shared modules across programmes.

**Why this comes before the next step:** the common jobs pool is compared against modules, not raw CSV rows, so the module tables must already have consistent programme membership and weights before any job comparison is constructed.


In [ ]:
def load_and_preprocess_parquet(
    parquet_path,
    dataset_name,
    id_col,
    embedding_cols,
    keep_cols=None,
    uppercase_id=False,
) -> pd.DataFrame:
    df = pd.read_parquet(parquet_path).copy()
    df.columns = df.columns.str.strip().str.lower()

    if isinstance(embedding_cols, str):
        embedding_cols = [embedding_cols]

    for col in [id_col, *embedding_cols]:
        col = col.lower()
        if col not in df.columns:
            raise KeyError(
                f"'{col}' column not found in {dataset_name}. "
                f"Available columns: {df.columns.tolist()}"
            )

    id_col = id_col.lower()
    embedding_cols = [col.lower() for col in embedding_cols]

    df[id_col] = df[id_col].astype(str).str.strip()
    if uppercase_id:
        df[id_col] = df[id_col].str.upper()

    if keep_cols is not None:
        keep_cols = [col.lower() for col in keep_cols]
        missing_keep_cols = [col for col in keep_cols if col not in df.columns]
        if missing_keep_cols:
            raise KeyError(
                f"Columns {missing_keep_cols} not found in {dataset_name}. "
                f"Available columns: {df.columns.tolist()}"
            )
        df = df[keep_cols].copy()

    return df


def load_curriculum_csv(csv_path, dataset_name):
    df = pd.read_csv(csv_path).copy()
    df.columns = df.columns.str.strip().str.lower()
    required_cols = ["course", "type", "code", "weight"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise KeyError(
            f"Columns {missing_cols} not found in {dataset_name}. "
            f"Available columns: {df.columns.tolist()}"
        )
    return df


def _standardize_module_keys(df):
    out = df.copy()
    out.columns = out.columns.str.strip().str.lower()
    if "course" in out.columns:
        out["course"] = out["course"].astype(str).str.strip().str.lower()
    if "type" in out.columns:
        out["type"] = out["type"].astype(str).str.strip().str.lower()
    if "code" in out.columns:
        out["code"] = out["code"].astype(str).str.strip().str.upper()
    return out


def merge_curriculum_metadata(embedded_df, curriculum_df, dataset_name):
    embedded = _standardize_module_keys(embedded_df)
    curriculum = _standardize_module_keys(curriculum_df)

    merge_cols = [col for col in ["course", "type", "code"] if col in embedded.columns and col in curriculum.columns]
    if merge_cols != ["course", "type", "code"]:
        raise KeyError(
            f"{dataset_name}: expected ['course', 'type', 'code'] in both embedded and curriculum tables."
        )

    curriculum_keep = merge_cols + [col for col in ["weight", "description"] if col in curriculum.columns]
    curriculum_small = curriculum[curriculum_keep].copy()

    merged = embedded.merge(
        curriculum_small,
        how="left",
        on=merge_cols,
        suffixes=("", "_curriculum")
    )

    if "description" in merged.columns and "description_curriculum" in merged.columns:
        merged["description"] = (
            merged["description"]
            .fillna("")
            .astype(str)
            .str.strip()
        )
        curriculum_desc = merged["description_curriculum"].fillna("").astype(str).str.strip()
        merged["description"] = merged["description"].where(merged["description"] != "", curriculum_desc)
        merged = merged.drop(columns=["description_curriculum"])

    merged["weight"] = pd.to_numeric(merged.get("weight"), errors="coerce")
    merged["weight"] = merged["weight"].where(merged["weight"] > 0, np.nan).fillna(1.0)

    return merged


module_keep_cols = ["course", "type", "code", "title", "description", "weight", "skill_embedding"]

raw_module_tables = {}
curriculum_tables = {}
merged_module_tables = {}
ingest_rows = []

for school in SCHOOLS:
    embedded_df = load_and_preprocess_parquet(
        parquet_path=MODULE_PARQUET_PATHS[school],
        dataset_name=f"{school.upper()} embedded modules",
        id_col="code",
        embedding_cols="skill_embedding",
        keep_cols=None,
        uppercase_id=True,
    )
    curriculum_df = load_curriculum_csv(
        CURRICULUM_CSV_PATHS[school],
        dataset_name=f"{school.upper()} curriculum csv",
    )
    merged_df = merge_curriculum_metadata(
        embedded_df=embedded_df,
        curriculum_df=curriculum_df,
        dataset_name=school.upper(),
    )
    raw_module_tables[school] = embedded_df
    curriculum_tables[school] = curriculum_df
    merged_module_tables[school] = merged_df

    ingest_rows.append(
        {
            "school": SCHOOL_DISPLAY[school],
            "embedded_rows": len(embedded_df),
            "curriculum_rows": len(curriculum_df),
            "merged_rows": len(merged_df),
            "missing_weight_rows_after_merge": int(merged_df["weight"].isna().sum()),
        }
    )


## Step 2: Construct the Common Job Pool

**Purpose:** define one shared labour-market job pool for broad employability analysis.

**What is done:** load the jobs parquet, retain jobs with valid embeddings, apply the employment-type filter, and derive `ssic_industry_family_2d` from `ssiccode` by stripping non-digits and taking the first two digits.

**Why this comes before the next step:** once the common job pool is fixed, every module can be evaluated against the same market reference set. Result 2 therefore reflects employer industry using SSIC, not occupation.


In [ ]:
def _get_optional_column(df: pd.DataFrame, candidates: list[str]):
    col_map = {col.lower(): col for col in df.columns}
    for candidate in candidates:
        if candidate.lower() in col_map:
            return col_map[candidate.lower()]
    return None


def _is_valid_embedding(x) -> bool:
    if x is None:
        return False
    arr = np.asarray(x)
    return arr.ndim == 1 and arr.size > 0 and np.all(np.isfinite(arr))


def _normalize_rows(matrix: np.ndarray, matrix_name: str = "matrix") -> np.ndarray:
    matrix = np.asarray(matrix, dtype=float)
    if matrix.ndim != 2:
        raise ValueError(f"{matrix_name} must be a 2D array, got shape {matrix.shape}")
    row_norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    if np.any(row_norms == 0):
        zero_count = int((row_norms == 0).sum())
        raise ValueError(f"{matrix_name} contains {zero_count} zero-norm rows.")
    return matrix / row_norms


def _employment_type_matches(value, allowed_types=None):
    if allowed_types is None:
        allowed_types = {"full time", "permanent", "contract"}
    if value is None:
        return False
    if isinstance(value, np.ndarray):
        values = [str(v).strip().lower() for v in value.tolist()]
        return any(v in allowed_types for v in values)
    if isinstance(value, (list, tuple, set)):
        values = [str(v).strip().lower() for v in value]
        return any(v in allowed_types for v in values)
    if pd.isna(value):
        return False
    value_str = str(value).strip()
    if value_str.startswith("[") and value_str.endswith("]"):
        try:
            parsed = ast.literal_eval(value_str)
            if isinstance(parsed, np.ndarray):
                parsed = parsed.tolist()
            if isinstance(parsed, (list, tuple, set)):
                values = [str(v).strip().lower() for v in parsed]
                return any(v in allowed_types for v in values)
        except (ValueError, SyntaxError):
            pass
    return value_str.lower() in allowed_types


def derive_ssic_industry_family_2d(ssic_value):
    if ssic_value is None or pd.isna(ssic_value):
        return np.nan

    digits = re.sub(r"\D", "", str(ssic_value))

    if len(digits) >= 2:
        return digits[:2]

    return np.nan

def prepare_module_pool_from_preprocessed(
    modules_df: pd.DataFrame,
    embedding_col: str = "skill_embedding",
    dataset_name: str = "modules_df",
) -> pd.DataFrame:
    df = _standardize_module_keys(modules_df)
    embedding_col = embedding_col.lower()

    if embedding_col not in df.columns:
        raise KeyError(
            f"'{embedding_col}' not found in {dataset_name}. "
            f"Available columns: {df.columns.tolist()}"
        )

    dedupe_cols = [col for col in ["course", "code", "type", "weight"] if col in df.columns]
    exact_duplicate_count = int(df.duplicated(subset=dedupe_cols, keep="first").sum()) if dedupe_cols else 0
    if dedupe_cols:
        df = df.drop_duplicates(subset=dedupe_cols, keep="first").copy()

    valid_mask = df[embedding_col].apply(_is_valid_embedding)
    dropped_invalid = int((~valid_mask).sum())
    if dropped_invalid > 0:
        warnings.warn(f"{dataset_name}: dropped {dropped_invalid} rows with invalid embeddings.")

    df = df[valid_mask].reset_index(drop=True)
    df.attrs["exact_duplicate_rows_removed"] = exact_duplicate_count

    if "weight" in df.columns:
        df["weight"] = pd.to_numeric(df["weight"], errors="coerce").where(lambda s: s > 0, np.nan).fillna(1.0)
        df["curriculum_weight"] = df["weight"].astype(float)
    else:
        df["curriculum_weight"] = 1.0

    if df.empty:
        warnings.warn(f"{dataset_name}: no valid module rows remaining after filtering.")

    return df


def prepare_job_pool_from_preprocessed(
    jobs_df: pd.DataFrame,
    embedding_col: str = "embedding_mpnet",
    restrict_employment_types: bool = True,
) -> pd.DataFrame:
    df = jobs_df.copy()
    df.columns = df.columns.str.strip().str.lower()
    embedding_col = embedding_col.lower()

    if embedding_col not in df.columns:
        raise KeyError(
            f"'{embedding_col}' not found in jobs_df. "
            f"Available columns: {df.columns.tolist()}"
        )

    if "uuid" in df.columns:
        df["uuid"] = df["uuid"].astype(str).str.strip()
        df = df.drop_duplicates(subset=["uuid"], keep="first").reset_index(drop=True)

    if restrict_employment_types:
        employment_type_col = _get_optional_column(
            df,
            ["employment_type", "employmenttype", "employmenttypes", "type", "job_type"]
        )
        if employment_type_col is not None:
            allowed_types = {"full time", "permanent", "contract"}
            df = df[
                df[employment_type_col].apply(
                    lambda x: _employment_type_matches(x, allowed_types=allowed_types)
                )
            ].reset_index(drop=True)
        else:
            warnings.warn(
                "No employment type column found in jobs_df. Skipping employment type filtering."
            )

    valid_mask = df[embedding_col].apply(_is_valid_embedding)
    dropped_invalid = int((~valid_mask).sum())
    if dropped_invalid > 0:
        warnings.warn(f"jobs_df: dropped {dropped_invalid} rows with invalid embeddings.")
    df = df[valid_mask].reset_index(drop=True)

    if "ssiccode" in df.columns:
        df["ssic_industry_family_2d"] = df["ssiccode"].apply(derive_ssic_industry_family_2d)
        df.attrs["industry_source"] = "ssiccode"
    else:
        warnings.warn(
            "jobs_df does not contain 'ssiccode'. Industry family diagnostics will be missing."
        )
        df["ssic_industry_family_2d"] = np.nan
        df.attrs["industry_source"] = "missing"

    if df.empty:
        warnings.warn("jobs_df: no valid job rows remaining after filtering.")

    return df


jobs_raw_df = load_and_preprocess_parquet(
    parquet_path=jobs_path,
    dataset_name="Jobs",
    id_col="uuid",
    embedding_cols="embedding_mpnet",
    keep_cols=None,
    uppercase_id=False,
)

job_pool_df = prepare_job_pool_from_preprocessed(
    jobs_df=jobs_raw_df,
    embedding_col="embedding_mpnet",
    restrict_employment_types=True,
)

industry_source_used = job_pool_df.attrs.get("industry_source", "missing")
job_pool_summary_df = pd.DataFrame(
    {
        "item": [
            "Raw jobs loaded",
            "Jobs retained in common pool",
            "Unique job ids in pool",
            "Industry source used",
            "Distinct 2-digit industry families retained",
        ],
        "value": [
            len(jobs_raw_df),
            len(job_pool_df),
            job_pool_df["uuid"].nunique() if "uuid" in job_pool_df.columns else np.nan,
            industry_source_used,
            job_pool_df["ssic_industry_family_2d"].dropna().astype(str).nunique(),
        ],
    }
)

print("Common job pool summary")
display(job_pool_summary_df)


# Part B — Build the Similarity Objects


## Step 3: Compute Module–Job Similarity Matrices

**Purpose:** construct the raw semantic comparison objects that link curriculum text to the labour market.

**What is done:** validate module embeddings, prepare clean module pools, normalize embeddings, compute cosine similarity between each module and each job, and store one module-job matrix per university.

**Why this comes before the next step:** threshold-free module features are computed from these similarity rows, so the raw similarity objects must exist before any validated module scoring can begin.


In [ ]:
def compute_cosine_similarity_matrix(module_embeddings: np.ndarray, job_embeddings: np.ndarray) -> np.ndarray:
    if module_embeddings.ndim != 2 or job_embeddings.ndim != 2:
        raise ValueError("Both module_embeddings and job_embeddings must be 2D arrays.")
    if module_embeddings.shape[1] != job_embeddings.shape[1]:
        raise ValueError(
            f"Embedding dimension mismatch: modules={module_embeddings.shape[1]}, jobs={job_embeddings.shape[1]}"
        )
    module_embeddings = _normalize_rows(module_embeddings, matrix_name="module embedding matrix")
    job_embeddings = _normalize_rows(job_embeddings, matrix_name="job embedding matrix")
    return module_embeddings @ job_embeddings.T


def build_module_job_similarity_matrix(
    modules_df: pd.DataFrame,
    job_pool_df: pd.DataFrame,
    module_embedding_col: str = "skill_embedding",
    job_embedding_col: str = "embedding_mpnet",
    dataset_name: str = "modules_df",
):
    clean_modules_df = prepare_module_pool_from_preprocessed(
        modules_df=modules_df,
        embedding_col=module_embedding_col,
        dataset_name=dataset_name,
    )

    if clean_modules_df.empty or job_pool_df.empty:
        return pd.DataFrame(), np.empty((0, 0), dtype=float)

    module_embeddings = np.stack(clean_modules_df[module_embedding_col].values)
    job_embeddings = np.stack(job_pool_df[job_embedding_col].values)

    similarity_matrix = compute_cosine_similarity_matrix(
        module_embeddings=module_embeddings,
        job_embeddings=job_embeddings,
    )

    clean_modules_df = clean_modules_df.reset_index(drop=True).copy()
    clean_modules_df["_module_idx"] = clean_modules_df.index
    return clean_modules_df, similarity_matrix


def plot_similarity_distributions_2panel(
    similarity_matrix,
    school_label,
    bins_pairwise=60,
    bins_max=40,
    show_percent=True,
):
    school_key = school_label.strip().lower()
    primary_color = school_colors[school_key]["primary"]
    secondary_color = school_colors[school_key]["secondary"]

    similarity_matrix = np.asarray(similarity_matrix, dtype=float)
    flat_scores = similarity_matrix.ravel()
    max_scores = similarity_matrix.max(axis=1)

    pairwise_weights = np.ones_like(flat_scores, dtype=float) / flat_scores.size
    module_weights = np.ones_like(max_scores, dtype=float) / max_scores.size

    fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

    axes[0].hist(
        flat_scores,
        bins=bins_pairwise,
        weights=pairwise_weights,
        color=primary_color,
        edgecolor="black",
        alpha=0.9,
    )
    axes[0].set_title(f"{school_label}: All Module-Job Similarity Scores")
    axes[0].set_xlabel("Cosine Similarity")
    axes[0].set_ylabel("Proportion of Module-Job Pairs")
    axes[0].grid(alpha=0.2)

    axes[1].hist(
        max_scores,
        bins=bins_max,
        weights=module_weights,
        color=secondary_color,
        edgecolor="black",
        alpha=0.9,
    )
    axes[1].set_title(f"{school_label}: Module Maximum Similarity")
    axes[1].set_xlabel("Maximum Cosine Similarity Across All Jobs")
    axes[1].set_ylabel("Proportion of Modules")
    axes[1].grid(alpha=0.2)

    if show_percent:
        axes[0].yaxis.set_major_formatter(PercentFormatter(xmax=1))
        axes[1].yaxis.set_major_formatter(PercentFormatter(xmax=1))

    plt.tight_layout()
    plt.show()


school_module_inputs = {}
school_similarity_matrices = {}
similarity_summary_rows = []

for school in SCHOOLS:
    modules_clean_df, similarity_matrix = build_module_job_similarity_matrix(
        modules_df=merged_module_tables[school],
        job_pool_df=job_pool_df,
        module_embedding_col="skill_embedding",
        job_embedding_col="embedding_mpnet",
        dataset_name=f"{school.upper()} merged modules",
    )

    school_module_inputs[school] = modules_clean_df
    school_similarity_matrices[school] = similarity_matrix
    similarity_summary_rows.append(
        {
            "school": SCHOOL_DISPLAY[school],
            "clean_module_rows": len(modules_clean_df),
            "num_programmes": int(modules_clean_df["course"].nunique()),
            "similarity_matrix_shape": similarity_matrix.shape,
            "exact_duplicate_membership_rows_removed": modules_clean_df.attrs.get("exact_duplicate_rows_removed", 0),
        }
    )

similarity_summary_df = pd.DataFrame(similarity_summary_rows)
print("Module pool and similarity-matrix summary")
display(similarity_summary_df)

for school, similarity_matrix in school_similarity_matrices.items():
    print(f"{SCHOOL_DISPLAY[school]} similarity matrix shape: {similarity_matrix.shape}")
    plot_similarity_distributions_2panel(similarity_matrix=similarity_matrix, school_label=SCHOOL_DISPLAY[school])


## Step 4: Construct Threshold-Free Module Features

**Purpose:** summarize each module’s similarity row using features that do not depend on any reach threshold.

**What is done:** compute `s_jobs`, `max_job_similarity`, `mean_similarity_all_jobs`, `s_level`, `s_prereqs`, and prerequisite unlock counts where available.

**Why this comes before the next step:** these features are safe inputs to the Spearman stage because they do not use any threshold-dependent reach metric, which preserves the non-circular ordering of the methodology.


In [ ]:
def load_prereq_graph(json_path):
    if json_path is None:
        return {}
    path = Path(json_path)
    if not path.exists():
        return {}
    with path.open("r") as f:
        raw_graph = json.load(f)
    graph = {}
    for key, values in raw_graph.items():
        key_clean = str(key).strip().upper()
        graph[key_clean] = [str(v).strip().upper() for v in values]
    return graph


def build_prereq_decay_lookup(ols_weights_df, default_lambda=0.10):
    if ols_weights_df is None or ols_weights_df.empty:
        return {school: default_lambda for school in SCHOOLS}
    df = ols_weights_df.copy()
    df.columns = df.columns.str.strip().str.lower()
    out = {school: default_lambda for school in SCHOOLS}
    if "university" not in df.columns or "lambda" not in df.columns:
        return out
    for _, row in df.iterrows():
        school = str(row["university"]).strip().lower()
        if school in out and pd.notna(row["lambda"]):
            out[school] = float(row["lambda"])
    return out


def extract_module_level_from_code(code):
    if code is None or pd.isna(code):
        return np.nan
    text = str(code).strip().upper()
    digit_groups = re.findall(r"\d+", text)
    if not digit_groups:
        return np.nan
    candidate = digit_groups[-1] if "-" in text else max(digit_groups, key=len)
    if len(candidate) == 0:
        return np.nan
    try:
        level = int(candidate[0])
    except ValueError:
        return np.nan
    if level <= 0:
        return np.nan
    return level


def _normalize_series(series, fill_value=0.0):
    values = pd.to_numeric(series, errors="coerce")
    min_val = values.min()
    max_val = values.max()
    if pd.isna(min_val) or pd.isna(max_val):
        return pd.Series(fill_value, index=series.index, dtype=float)
    if np.isclose(min_val, max_val):
        return pd.Series(fill_value if pd.notna(fill_value) else 0.5, index=series.index, dtype=float)
    return (values - min_val) / (max_val - min_val)


def get_descendant_depths(graph, start_code):
    start_code = str(start_code).strip().upper()
    if start_code not in graph:
        return {}

    seen_depths = {}
    queue = deque([(start_code, 0)])

    while queue:
        node, depth = queue.popleft()
        for child in graph.get(node, []):
            next_depth = depth + 1
            if child not in seen_depths or next_depth < seen_depths[child]:
                seen_depths[child] = next_depth
                queue.append((child, next_depth))

    return seen_depths


def compute_module_feature_df(
    modules_clean_df,
    similarity_matrix,
    top_k=100,
    prereq_graph=None,
    prereq_decay_lambda=0.10,
):
    if modules_clean_df.empty:
        return modules_clean_df.copy()

    modules = modules_clean_df.copy().reset_index(drop=True)
    n_jobs = similarity_matrix.shape[1]
    k = min(max(int(top_k), 1), n_jobs)

    if k == n_jobs:
        top_k_values = similarity_matrix
    else:
        top_k_values = np.partition(similarity_matrix, n_jobs - k, axis=1)[:, -k:]

    modules["s_jobs"] = top_k_values.mean(axis=1)
    modules["max_job_similarity"] = similarity_matrix.max(axis=1)
    modules["mean_similarity_all_jobs"] = similarity_matrix.mean(axis=1)

    raw_level = modules["code"].map(extract_module_level_from_code) if "code" in modules.columns else np.nan
    modules["raw_level"] = raw_level
    modules["s_level"] = _normalize_series(raw_level, fill_value=0.0)

    modules["num_unlocks"] = 0
    modules["s_prereqs"] = 0.0

    if prereq_graph:
        code_to_sjobs = (
            modules.groupby("code", dropna=False)["s_jobs"]
            .mean()
            .to_dict()
        )
        num_unlocks_list = []
        prereq_raw_list = []

        for code_value in modules["code"].astype(str):
            depths = get_descendant_depths(prereq_graph, code_value)
            num_unlocks = len(depths)

            weighted_scores = []
            for child_code, depth in depths.items():
                child_sjobs = code_to_sjobs.get(child_code)
                if child_sjobs is None or pd.isna(child_sjobs):
                    continue
                decay = prereq_decay_lambda ** max(depth - 1, 0)
                weighted_scores.append(float(child_sjobs) * decay)

            downstream_signal = float(np.mean(weighted_scores)) if weighted_scores else 0.0
            prereq_raw = np.log1p(num_unlocks) * downstream_signal if num_unlocks > 0 else 0.0

            num_unlocks_list.append(num_unlocks)
            prereq_raw_list.append(prereq_raw)

        modules["num_unlocks"] = num_unlocks_list
        modules["s_prereqs"] = _normalize_series(pd.Series(prereq_raw_list, index=modules.index), fill_value=0.0)

    modules["curriculum_weight"] = pd.to_numeric(
        modules.get("curriculum_weight", modules.get("weight", 1.0)),
        errors="coerce",
    ).where(lambda s: s > 0, np.nan).fillna(1.0)

    keep_front = [col for col in ["course", "type", "code", "curriculum_weight", "weight"] if col in modules.columns]
    ordered_cols = keep_front + [
        col for col in [
            "s_jobs",
            "max_job_similarity",
            "mean_similarity_all_jobs",
            "raw_level",
            "s_level",
            "num_unlocks",
            "s_prereqs",
            "_module_idx",
        ]
        if col in modules.columns and col not in keep_front
    ]
    remaining_cols = [col for col in modules.columns if col not in ordered_cols]
    return modules[ordered_cols + remaining_cols]


ols_weights_df = pd.read_csv(ols_weights_path)
prereq_graphs = {school: load_prereq_graph(PREREQ_GRAPH_PATHS[school]) for school in SCHOOLS}
prereq_decay_lookup = build_prereq_decay_lookup(ols_weights_df=ols_weights_df, default_lambda=0.10)

school_threshold_free_module_feature_dfs = {}
threshold_free_summary_rows = []

for school_key, modules_clean_df in school_module_inputs.items():
    module_feature_df = compute_module_feature_df(
        modules_clean_df=modules_clean_df,
        similarity_matrix=school_similarity_matrices[school_key],
        top_k=TOP_K,
        prereq_graph=prereq_graphs.get(school_key, {}),
        prereq_decay_lambda=prereq_decay_lookup.get(school_key, 0.10),
    )
    school_threshold_free_module_feature_dfs[school_key] = module_feature_df
    threshold_free_summary_rows.append(
        {
            "school": SCHOOL_DISPLAY[school_key],
            "mean_s_jobs": float(module_feature_df["s_jobs"].mean()),
            "mean_max_job_similarity": float(module_feature_df["max_job_similarity"].mean()),
            "mean_s_level": float(module_feature_df["s_level"].mean()),
            "mean_s_prereqs": float(module_feature_df["s_prereqs"].mean()),
        }
    )

threshold_free_feature_summary_df = pd.DataFrame(threshold_free_summary_rows)
print("Threshold-free module feature summary")
display(threshold_free_feature_summary_df)

threshold_free_preview_df = pd.concat(
    [
        school_threshold_free_module_feature_dfs[school_name][
            ["course", "type", "code", "s_jobs", "max_job_similarity", "s_level", "s_prereqs"]
        ]
        .assign(school=SCHOOL_DISPLAY[school_name])
        .sort_values("s_jobs", ascending=False)
        .head(6)
        for school_name in SCHOOLS
    ],
    ignore_index=True,
)

print("Top threshold-free modules by s_jobs")
display(threshold_free_preview_df)


## Step 5: Build Spearman-Informed Module Importance Scores

**Purpose:** translate validated degree-level association evidence into a module-level importance prior.

**What is done:** load the Spearman correlation table, parse significance flags, build university-specific normalized absolute-Spearman weight maps, compute `spearman_module_score`, and create `effective_module_weight` at module level.

**Why this comes before the next step:** programme aggregation should reflect both curriculum structure and validated module-feature importance, so the Spearman-informed module scores must exist before modules are combined into programme-job vectors.


In [ ]:
spearman_df = pd.read_csv(spearman_path)

def parse_significant_flag(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or pd.isna(value):
        return False
    value_str = str(value).strip().lower()
    return value_str in {"true", "1", "yes", "y", "significant"}


def get_spearman_weight_map(
    spearman_df,
    university,
    significant_only=True,
    use_absolute_rho=True,
):
    df = spearman_df.copy()
    df.columns = df.columns.str.strip().str.lower()

    required_cols = ["university", "feature", "spearman_rho"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise KeyError(f"Spearman table is missing required column(s): {missing_cols}")

    university = str(university).strip().lower()
    df["university"] = df["university"].astype(str).str.strip().str.lower()
    df = df[df["university"] == university].copy()

    if significant_only and "significant" in df.columns:
        df = df[df["significant"].apply(parse_significant_flag)].copy()

    if df.empty:
        return {}

    weights = pd.to_numeric(df["spearman_rho"], errors="coerce")
    if use_absolute_rho:
        weights = weights.abs()

    df = df.assign(weight_value=weights).dropna(subset=["weight_value"])
    df = df[df["weight_value"] > 0].copy()

    if df.empty:
        return {}

    weight_sum = df["weight_value"].sum()
    if weight_sum <= 0:
        return {}

    df["normalized_weight"] = df["weight_value"] / weight_sum
    return dict(zip(df["feature"], df["normalized_weight"]))


def add_spearman_module_importance(
    module_feature_df,
    spearman_df,
    university,
    feature_map=None,
    normalize_score=True,
):
    if feature_map is None:
        feature_map = {
            "avg_s_jobs": "s_jobs",
            "avg_s_level": "s_level",
            "avg_s_prereqs": "s_prereqs",
        }

    out = module_feature_df.copy()
    out.columns = out.columns.str.strip().str.lower()

    weight_map = get_spearman_weight_map(
        spearman_df=spearman_df,
        university=university,
        significant_only=True,
        use_absolute_rho=True,
    )

    usable_pairs = [
        (spearman_feature, module_feature_col, weight_map[spearman_feature])
        for spearman_feature, module_feature_col in feature_map.items()
        if spearman_feature in weight_map and module_feature_col in out.columns
    ]

    out["spearman_module_score"] = 1.0
    out["spearman_weight_source"] = "curriculum_only_fallback"
    out["spearman_used_features"] = ""

    if usable_pairs:
        normalized_feature_frame = pd.DataFrame(index=out.index)
        for _, module_feature_col, _ in usable_pairs:
            normalized_feature_frame[module_feature_col] = _normalize_series(
                out[module_feature_col],
                fill_value=0.0,
            )

        raw_score = np.zeros(len(out), dtype=float)
        for _, module_feature_col, feature_weight in usable_pairs:
            raw_score += normalized_feature_frame[module_feature_col].fillna(0).to_numpy() * feature_weight

        raw_score = np.clip(raw_score, a_min=1e-6, a_max=None)
        if normalize_score:
            mean_score = float(np.nanmean(raw_score)) if len(raw_score) > 0 else 1.0
            mean_score = mean_score if mean_score > 0 else 1.0
            score = raw_score / mean_score
        else:
            score = raw_score

        score = np.clip(score, a_min=0.25, a_max=None)
        out["spearman_module_score"] = score
        out["spearman_weight_source"] = "spearman_validated"
        out["spearman_used_features"] = ", ".join(module_feature_col for _, module_feature_col, _ in usable_pairs)

    out["effective_module_weight"] = (
        pd.to_numeric(out.get("curriculum_weight", 1.0), errors="coerce").fillna(1.0)
        * pd.to_numeric(out["spearman_module_score"], errors="coerce").fillna(1.0)
    )
    return out


def build_spearman_weight_summary(spearman_df):
    rows = []
    for school in SCHOOLS:
        weight_map = get_spearman_weight_map(spearman_df, school, significant_only=True, use_absolute_rho=True)
        if not weight_map:
            rows.append(
                {
                    "school": SCHOOL_DISPLAY[school],
                    "feature": "(fallback to curriculum weights only)",
                    "normalized_abs_spearman_weight": np.nan,
                }
            )
            continue

        for feature, weight in weight_map.items():
            rows.append(
                {
                    "school": SCHOOL_DISPLAY[school],
                    "feature": feature,
                    "normalized_abs_spearman_weight": weight,
                }
            )
    return pd.DataFrame(rows)


school_module_feature_dfs = {}
module_importance_summary_rows = []

for school_key, module_feature_df in school_threshold_free_module_feature_dfs.items():
    scored_df = add_spearman_module_importance(
        module_feature_df=module_feature_df,
        spearman_df=spearman_df,
        university=school_key,
        feature_map={
            "avg_s_jobs": "s_jobs",
            "avg_s_level": "s_level",
            "avg_s_prereqs": "s_prereqs",
        },
        normalize_score=True,
    )
    school_module_feature_dfs[school_key] = scored_df
    module_importance_summary_rows.append(
        {
            "school": SCHOOL_DISPLAY[school_key],
            "mean_curriculum_weight": float(scored_df["curriculum_weight"].mean()),
            "mean_spearman_module_score": float(scored_df["spearman_module_score"].mean()),
            "spearman_weight_source": scored_df["spearman_weight_source"].mode().iloc[0],
            "used_feature_signature": scored_df["spearman_used_features"].mode().iloc[0],
        }
    )

spearman_weight_summary_df = build_spearman_weight_summary(spearman_df)
module_importance_summary_df = pd.DataFrame(module_importance_summary_rows)

print("Spearman weighting summary")
display(module_importance_summary_df)
display(spearman_weight_summary_df)

top_module_score_df = pd.concat(
    [
        school_module_feature_dfs[school_name][
            ["course", "type", "code", "s_jobs", "s_level", "s_prereqs", "spearman_module_score"]
        ]
        .assign(school=SCHOOL_DISPLAY[school_name])
        .sort_values("spearman_module_score", ascending=False)
        .head(8)
        for school_name in SCHOOLS
    ],
    ignore_index=True,
)

print("Highest-scoring modules under the Spearman importance stage")
display(top_module_score_df)


## Step 6: Aggregate Module Scores to Programme–Job Matrices

**Purpose:** move from module-level evidence to programme-level employability objects.

**What is done:** combine curriculum weights and Spearman-informed module scores into `effective_module_weight = curriculum_weight * spearman_module_score`, normalize weights within programme, and aggregate module-job rows into final programme-job vectors.

**Why this comes before the next step:** threshold calibration must operate on the final Spearman-weighted programme-job matrices, not on earlier unweighted similarities or module-level rows.


In [ ]:
def _resolve_module_weights(
    modules_df,
    weight_col="curriculum_weight",
    type_col="type",
    type_weight_map=None,
    default_weight=1.0,
    spearman_score_col=None,
):
    modules = modules_df.copy()
    modules.columns = modules.columns.str.strip().str.lower()

    weight_col = weight_col.lower() if weight_col is not None else None
    type_col = type_col.lower() if type_col is not None else None
    spearman_score_col = spearman_score_col.lower() if spearman_score_col is not None else None

    if weight_col is not None and weight_col in modules.columns:
        weights = pd.to_numeric(modules[weight_col], errors="coerce")
        weights = weights.where(weights > 0, np.nan).fillna(default_weight)
    else:
        weights = pd.Series(default_weight, index=modules.index, dtype=float)

    if type_weight_map is not None and type_col is not None and type_col in modules.columns:
        module_types = modules[type_col].fillna("").astype(str).str.strip().str.lower()
        type_multipliers = module_types.map(type_weight_map).fillna(1.0)
        weights = weights * type_multipliers

    if spearman_score_col is not None and spearman_score_col in modules.columns:
        spearman_scores = pd.to_numeric(modules[spearman_score_col], errors="coerce").fillna(1.0)
        spearman_scores = spearman_scores.where(spearman_scores > 0, 1.0)
        weights = weights * spearman_scores

    return weights.astype(float)


def build_programme_job_matrix(
    modules_clean_df,
    similarity_matrix,
    job_pool_df,
    programme_col="course",
    weight_col="curriculum_weight",
    type_col="type",
    type_weight_map=None,
    default_weight=1.0,
    normalize_weights_within_programme=True,
    job_id_col="uuid",
    spearman_score_col=None,
):
    modules = modules_clean_df.copy()
    modules.columns = modules.columns.str.strip().str.lower()

    programme_col = programme_col.lower()
    weight_col = weight_col.lower() if weight_col is not None else None
    type_col = type_col.lower() if type_col is not None else None
    spearman_score_col = spearman_score_col.lower() if spearman_score_col is not None else None

    if programme_col not in modules.columns:
        raise KeyError(f"modules_clean_df must contain '{programme_col}'.")
    if len(modules) != similarity_matrix.shape[0]:
        raise ValueError(
            "Row mismatch: modules_clean_df and similarity_matrix must have the same number of module rows."
        )

    modules[programme_col] = modules[programme_col].fillna("").astype(str).str.strip().str.lower()
    modules = modules[modules[programme_col] != ""].copy().reset_index(drop=True)
    modules["_module_idx"] = modules.get("_module_idx", modules.index)

    modules["_effective_weight"] = _resolve_module_weights(
        modules_df=modules,
        weight_col=weight_col,
        type_col=type_col,
        type_weight_map=type_weight_map,
        default_weight=default_weight,
        spearman_score_col=spearman_score_col,
    )

    job_meta = job_pool_df.copy()
    job_meta.columns = job_meta.columns.str.strip().str.lower()
    if job_id_col and job_id_col in job_meta.columns:
        job_labels = job_meta[job_id_col].astype(str).tolist()
    elif "uuid" in job_meta.columns:
        job_labels = job_meta["uuid"].astype(str).tolist()
    elif "title" in job_meta.columns:
        job_labels = job_meta["title"].astype(str).tolist()
    else:
        job_labels = [f"job_{i}" for i in range(similarity_matrix.shape[1])]

    programme_vectors = {}
    programme_metadata_rows = []

    for programme, group in modules.groupby(programme_col, sort=True):
        module_row_idx = group["_module_idx"].to_numpy(dtype=int)
        module_weights = group["_effective_weight"].to_numpy(dtype=float)
        raw_weight_sum = float(module_weights.sum())

        if normalize_weights_within_programme:
            if raw_weight_sum <= 0:
                module_weights = np.repeat(1 / len(module_weights), len(module_weights))
            else:
                module_weights = module_weights / raw_weight_sum

        programme_vector = np.average(
            similarity_matrix[module_row_idx, :],
            axis=0,
            weights=module_weights,
        )
        programme_vectors[programme] = programme_vector

        type_series = group[type_col].fillna("").astype(str).str.strip().str.lower() if type_col in group.columns else pd.Series("", index=group.index)
        num_core_modules = int((type_series == "core").sum())
        num_elective_modules = int((type_series == "elective").sum())
        num_modules = len(group)
        core_share = num_core_modules / num_modules if num_modules > 0 else np.nan

        programme_metadata_rows.append(
            {
                "course": programme,
                "num_modules": num_modules,
                "raw_weight_sum": raw_weight_sum,
                "num_core_modules": num_core_modules,
                "num_elective_modules": num_elective_modules,
                "core_share": core_share,
                "mean_spearman_module_score": float(pd.to_numeric(group.get("spearman_module_score"), errors="coerce").mean()) if "spearman_module_score" in group.columns else np.nan,
                "mean_effective_module_weight": float(pd.to_numeric(group.get("_effective_weight"), errors="coerce").mean()),
            }
        )

    programme_job_matrix_df = pd.DataFrame.from_dict(
        programme_vectors,
        orient="index",
        columns=job_labels,
    )
    programme_job_matrix_df.index.name = "course"
    programme_job_matrix_df = programme_job_matrix_df.sort_index()

    programme_metadata_df = pd.DataFrame(programme_metadata_rows).sort_values("course").reset_index(drop=True)
    return programme_job_matrix_df, programme_metadata_df


school_programme_matrices = {}
school_programme_metadata = {}

for school in SCHOOLS:
    school_key = school
    programme_job_matrix_df, programme_metadata_df = build_programme_job_matrix(
        modules_clean_df=school_module_feature_dfs[school],
        similarity_matrix=school_similarity_matrices[school],
        job_pool_df=job_pool_df,
        programme_col="course",
        weight_col="curriculum_weight",
        type_col="type",
        type_weight_map=None,
        default_weight=1.0,
        normalize_weights_within_programme=True,
        job_id_col="uuid",
        spearman_score_col="spearman_module_score",
    )
    school_programme_matrices[school] = programme_job_matrix_df
    school_programme_metadata[school] = programme_metadata_df

def plot_programme_metadata_boxplots(programme_metadata_dict):
    metric_configs = [
        ("num_modules", "Number of modules per programme", "Num modules"),
        ("raw_weight_sum", "Raw effective weight sum", "Raw effective weight sum"),
        ("mean_spearman_module_score", "Mean Spearman module score", "Mean score"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
    school_order = SCHOOLS

    for ax, (metric_col, title, ylabel) in zip(axes, metric_configs):
        data, labels, colors = [], [], []
        for school_name in school_order:
            df = programme_metadata_dict[school_name]
            values = pd.to_numeric(df[metric_col], errors="coerce").dropna().to_numpy()
            data.append(values)
            labels.append(SCHOOL_DISPLAY[school_name])
            colors.append(school_colors[school_name.lower()]["primary"])

        bp = ax.boxplot(data, patch_artist=True, tick_labels=labels, widths=0.6)
        for patch, color in zip(bp["boxes"], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.85)
            patch.set_edgecolor("black")
        for median in bp["medians"]:
            median.set_color("black")
            median.set_linewidth(1.5)
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()


plot_programme_metadata_boxplots(school_programme_metadata)

print("Programme metadata preview")
for school_name in SCHOOLS:
    print(SCHOOL_DISPLAY[school_name])
    display(school_programme_metadata[school_name].head())


# Part C — Calibrate the Decision Rule


## Step 7: Define Programme-Level Evaluation Metrics

**Purpose:** define the post-threshold metrics before selecting the final decision rule.

**What is defined:**
- `reach_job_ratio`: the share of jobs whose programme-job similarity is at or above the threshold. Higher values indicate broader reachable labour-market coverage.
- `mean_similarity_all_jobs`, `top_k_mean_similarity`, and `max_similarity`: complementary strength metrics based on the full score distribution, the top `k=100` jobs, and the single strongest match. Higher values indicate stronger semantic alignment.
- `matched_industry_family_count`, `matched_industry_family_share`, and `normalized_industry_family_entropy`: breadth metrics based on reachable `ssic_industry_family_2d` groups. Higher values indicate wider spread across employer industries.
- robustness metrics such as `top1_module_share`, `top5_module_share`, and `normalized_effective_module_count`: diagnostics of whether observed alignment is broadly supported across the curriculum or concentrated in a small number of modules.

**Why this comes before the next step:** threshold calibration needs the metric definitions in place so candidate cutoffs can be compared consistently before the final results are reported.


In [ ]:
def _safe_entropy_from_counts(counts):
    counts = np.asarray(counts, dtype=float)
    counts = counts[counts > 0]
    if counts.size <= 1:
        return 0.0
    probs = counts / counts.sum()
    return float(-(probs * np.log(probs)).sum())


def compute_programme_reach_strength_breadth(
    programme_job_matrix_df,
    job_pool_df,
    similarity_threshold,
    top_k=100,
    family_col="ssic_industry_family_2d",
    school_name=None,
):
    if programme_job_matrix_df.empty:
        return pd.DataFrame()

    matrix = programme_job_matrix_df.to_numpy(dtype=float)
    programmes = programme_job_matrix_df.index.astype(str).tolist()
    n_jobs = matrix.shape[1]
    k = min(max(int(top_k), 1), n_jobs)

    reach_mask = matrix >= similarity_threshold
    reach_job_count = reach_mask.sum(axis=1)
    reach_job_ratio = reach_job_count / n_jobs

    mean_similarity_all_jobs = matrix.mean(axis=1)
    max_similarity = matrix.max(axis=1)

    if k == n_jobs:
        top_k_mean_similarity = mean_similarity_all_jobs.copy()
    else:
        top_k_values = np.partition(matrix, n_jobs - k, axis=1)[:, -k:]
        top_k_mean_similarity = top_k_values.mean(axis=1)

    metrics_df = pd.DataFrame(
        {
            "course": programmes,
            "reach_job_count": reach_job_count,
            "reach_job_ratio": reach_job_ratio,
            "mean_similarity_all_jobs": mean_similarity_all_jobs,
            "top_k_mean_similarity": top_k_mean_similarity,
            "max_similarity": max_similarity,
        }
    )

    family_values = job_pool_df[family_col].fillna("").astype(str).to_numpy()
    valid_family_mask = np.array([bool(x) and x.lower() != "nan" for x in family_values], dtype=bool)
    total_distinct_industry_families = len(set(family_values[valid_family_mask]))

    matched_industry_family_counts = []
    matched_industry_family_shares = []
    industry_family_entropies = []
    normalized_industry_family_entropies = []

    for i in range(len(programmes)):
        reached_idx = np.flatnonzero(reach_mask[i] & valid_family_mask)
        if reached_idx.size == 0:
            matched_industry_family_counts.append(0)
            matched_industry_family_shares.append(
                0.0 if total_distinct_industry_families > 0 else np.nan
            )
            industry_family_entropies.append(0.0)
            normalized_industry_family_entropies.append(0.0)
            continue

        reached_families = family_values[reached_idx]
        family_counts = pd.Series(reached_families).value_counts()
        family_count = int(family_counts.shape[0])
        family_share = (
            family_count / total_distinct_industry_families
            if total_distinct_industry_families > 0
            else np.nan
        )

        entropy_val = _safe_entropy_from_counts(family_counts.to_numpy())
        normalized_entropy_val = entropy_val / np.log(family_count) if family_count > 1 else 0.0

        matched_industry_family_counts.append(family_count)
        matched_industry_family_shares.append(family_share)
        industry_family_entropies.append(entropy_val)
        normalized_industry_family_entropies.append(normalized_entropy_val)

    metrics_df["matched_industry_family_count"] = matched_industry_family_counts
    metrics_df["matched_industry_family_share"] = matched_industry_family_shares
    metrics_df["industry_family_entropy"] = industry_family_entropies
    metrics_df["normalized_industry_family_entropy"] = normalized_industry_family_entropies

    if school_name is not None:
        metrics_df.insert(0, "school", school_name)

    return metrics_df


def compute_programme_robustness(
    module_diagnostics_df,
    similarity_matrix,
    programme_job_matrix_df,
    programme_col="course",
    weight_col="curriculum_weight",
    type_col="type",
    type_weight_map=None,
    default_weight=1.0,
    spearman_score_col="spearman_module_score",
    top_k_jobs=100,
    school_name=None,
):
    modules = module_diagnostics_df.copy()
    modules.columns = modules.columns.str.strip().str.lower()

    programme_col = programme_col.lower()
    weight_col = weight_col.lower() if weight_col is not None else None
    type_col = type_col.lower() if type_col is not None else None
    spearman_score_col = spearman_score_col.lower() if spearman_score_col is not None else None

    modules[programme_col] = modules[programme_col].fillna("").astype(str).str.strip().str.lower()
    modules = modules[modules[programme_col] != ""].copy()
    modules["_module_idx"] = modules.get("_module_idx", np.arange(len(modules)))

    modules["_effective_weight"] = _resolve_module_weights(
        modules_df=modules,
        weight_col=weight_col,
        type_col=type_col,
        type_weight_map=type_weight_map,
        default_weight=default_weight,
        spearman_score_col=spearman_score_col,
    )

    robustness_rows = []

    for programme in programme_job_matrix_df.index.astype(str):
        programme_key = str(programme).strip().lower()
        group = modules[modules[programme_col] == programme_key].copy()
        if group.empty:
            continue

        programme_vector = programme_job_matrix_df.loc[programme].to_numpy(dtype=float)
        n_jobs = programme_vector.shape[0]
        k = min(max(int(top_k_jobs), 1), n_jobs)

        if k == n_jobs:
            top_job_idx = np.arange(n_jobs)
        else:
            top_job_idx = np.argpartition(programme_vector, n_jobs - k)[-k:]
            top_job_idx = np.sort(top_job_idx)

        module_row_idx = group["_module_idx"].to_numpy(dtype=int)
        module_weights = group["_effective_weight"].to_numpy(dtype=float)
        module_strength = similarity_matrix[np.ix_(module_row_idx, top_job_idx)].mean(axis=1)
        weighted_contribution = module_weights * np.clip(module_strength, a_min=0, a_max=None)

        if weighted_contribution.sum() <= 0:
            contribution_shares = (
                module_weights / module_weights.sum()
                if module_weights.sum() > 0
                else np.repeat(1 / len(module_weights), len(module_weights))
            )
        else:
            contribution_shares = weighted_contribution / weighted_contribution.sum()

        contribution_shares = np.sort(contribution_shares)[::-1]
        n_modules = len(contribution_shares)
        top1_share = float(contribution_shares[:1].sum())
        top5_share = float(contribution_shares[:5].sum())
        rest_share = float(max(0.0, 1.0 - top5_share))

        hhi = float((contribution_shares ** 2).sum())
        effective_module_count = float(1.0 / hhi) if hhi > 0 else np.nan
        normalized_effective_module_count = (
            effective_module_count / n_modules if n_modules > 0 and pd.notna(effective_module_count) else np.nan
        )

        robustness_rows.append(
            {
                "course": programme_key,
                "top1_module_share": top1_share,
                "top5_module_share": top5_share,
                "rest_module_share": rest_share,
                "effective_module_count": effective_module_count,
                "normalized_effective_module_count": normalized_effective_module_count,
            }
        )

    robustness_df = pd.DataFrame(robustness_rows)
    if school_name is not None and not robustness_df.empty:
        robustness_df.insert(0, "school", school_name)
    return robustness_df


def build_programme_core_metrics(
    programme_job_matrix_df,
    job_pool_df,
    module_diagnostics_df,
    similarity_matrix,
    similarity_threshold,
    top_k=100,
    programme_col="course",
    weight_col="curriculum_weight",
    type_col="type",
    type_weight_map=None,
    default_weight=1.0,
    family_col="ssic_industry_family_2d",
    spearman_score_col="spearman_module_score",
    school_name=None,
):
    reach_strength_breadth_df = compute_programme_reach_strength_breadth(
        programme_job_matrix_df=programme_job_matrix_df,
        job_pool_df=job_pool_df,
        similarity_threshold=similarity_threshold,
        top_k=top_k,
        family_col=family_col,
        school_name=school_name,
    )

    robustness_df = compute_programme_robustness(
        module_diagnostics_df=module_diagnostics_df,
        similarity_matrix=similarity_matrix,
        programme_job_matrix_df=programme_job_matrix_df,
        programme_col=programme_col,
        weight_col=weight_col,
        type_col=type_col,
        type_weight_map=type_weight_map,
        default_weight=default_weight,
        spearman_score_col=spearman_score_col,
        top_k_jobs=top_k,
        school_name=school_name,
    )

    merge_cols = ["school", "course"] if school_name is not None else ["course"]
    return reach_strength_breadth_df.merge(robustness_df, how="left", on=merge_cols)


def assign_profile_labels_for_sweep(
    df,
    reach_col="reach_job_ratio",
    strength_col="top_k_mean_similarity",
    split_method="median",
):
    out = df.copy()
    if split_method == "median":
        x_split = out[reach_col].median()
        y_split = out[strength_col].median()
    elif split_method == "mean":
        x_split = out[reach_col].mean()
        y_split = out[strength_col].mean()
    else:
        raise ValueError("split_method must be 'median' or 'mean'.")

    conditions = [
        (out[reach_col] >= x_split) & (out[strength_col] >= y_split),
        (out[reach_col] < x_split) & (out[strength_col] >= y_split),
        (out[reach_col] >= x_split) & (out[strength_col] < y_split),
        (out[reach_col] < x_split) & (out[strength_col] < y_split),
    ]
    labels = [
        "Broad All-Rounders",
        "Niche Specialists",
        "Broad but Shallow",
        "Weak Overall",
    ]
    out["opportunity_profile"] = np.select(conditions, labels, default="Unclassified")
    return out

## Step 8: Threshold Calibration and Selection

**Purpose:** select one programme-job similarity threshold for the final decision rule.

**What is done:** compare the pooled distribution of final Spearman-weighted programme-job scores with a null baseline built from the same aggregation logic under shuffled job embeddings, then sweep candidate thresholds and evaluate retention, stability, and boundary behavior.

**Why a universal threshold is used:** every programme is scored against the same common job pool and the same final aggregation framework, so one screening indicator can be applied consistently across programmes.

**How the sweep supports selection:** the sweep shows how reach, industry breadth, near-zero coverage, and ranking stability change as the cutoff becomes stricter. This makes the final choice transparent rather than ad hoc.


In [ ]:
SCHOOL_PROGRAMME_MATRICES = {SCHOOL_DISPLAY[school]: df for school, df in school_programme_matrices.items()}
SCHOOL_MODULE_INPUTS = {SCHOOL_DISPLAY[school]: df for school, df in school_module_inputs.items()}
SCHOOL_MODULE_FEATURES = {SCHOOL_DISPLAY[school]: df for school, df in school_module_feature_dfs.items()}
SCHOOL_SIMILARITY_MATRICES = {SCHOOL_DISPLAY[school]: df for school, df in school_similarity_matrices.items()}


def stack_programme_job_scores(programme_job_matrix_df, school_name):
    out = programme_job_matrix_df.copy().stack().reset_index()
    out.columns = ["course", "job_id", "programme_job_similarity"]
    out.insert(0, "school", school_name)
    return out


all_programme_job_long_df = pd.concat(
    [
        stack_programme_job_scores(df, school_name)
        for school_name, df in SCHOOL_PROGRAMME_MATRICES.items()
    ],
    ignore_index=True,
)
real_programme_job_scores = all_programme_job_long_df["programme_job_similarity"].to_numpy(dtype=float)

real_distribution_summary_df = pd.DataFrame(
    {
        "statistic": ["count", "min", "p25", "median", "p75", "max"],
        "value": [
            len(real_programme_job_scores),
            np.min(real_programme_job_scores),
            np.quantile(real_programme_job_scores, 0.25),
            np.quantile(real_programme_job_scores, 0.50),
            np.quantile(real_programme_job_scores, 0.75),
            np.max(real_programme_job_scores),
        ],
    }
)

print("Pooled programme-job long dataframe shape:", all_programme_job_long_df.shape)
display(real_distribution_summary_df)

plt.figure(figsize=(10, 5.5))
plt.hist(
    real_programme_job_scores,
    bins=80,
    weights=np.ones_like(real_programme_job_scores) / len(real_programme_job_scores),
    edgecolor="black",
    alpha=0.85,
)
plt.title("Step 8A: Pooled Real Programme-Job Similarity Distribution", fontweight="bold")
plt.xlabel("Programme-Job Similarity")
plt.ylabel("Proportion of Programme-Job Pairs")
plt.gca().yaxis.set_major_formatter(PercentFormatter(xmax=1))
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
def _shuffle_embedding_dimensions_globally(embedding_matrix, rng):
    perm = rng.permutation(embedding_matrix.shape[1])
    return embedding_matrix[:, perm]


def simulate_null_programme_job_scores(
    school_modules_dict,
    job_pool_df,
    spearman_df,
    prereq_graphs_by_school,
    prereq_decay_by_school,
    n_iter=30,
    random_state=42,
    module_embedding_col="skill_embedding",
    job_embedding_col="embedding_mpnet",
    programme_col="course",
    weight_col="curriculum_weight",
    type_col="type",
    type_weight_map=None,
    default_weight=1.0,
    spearman_score_col="spearman_module_score",
    top_k=100,
):
    rng = np.random.default_rng(random_state)
    real_job_embeddings = np.stack(job_pool_df[job_embedding_col].values)
    null_score_chunks = []

    for b in range(n_iter):
        shuffled_job_embeddings = _shuffle_embedding_dimensions_globally(real_job_embeddings, rng)

        for school_name, modules_clean_df in school_modules_dict.items():
            school_key = str(school_name).strip().lower()
            module_embeddings = np.stack(modules_clean_df[module_embedding_col].values)

            null_similarity_matrix = compute_cosine_similarity_matrix(
                module_embeddings=module_embeddings,
                job_embeddings=shuffled_job_embeddings,
            )

            null_module_feature_df = compute_module_feature_df(
                modules_clean_df=modules_clean_df,
                similarity_matrix=null_similarity_matrix,
                top_k=top_k,
                prereq_graph=prereq_graphs_by_school.get(school_key, {}),
                prereq_decay_lambda=prereq_decay_by_school.get(school_key, 0.10),
            )

            null_module_feature_df = add_spearman_module_importance(
                module_feature_df=null_module_feature_df,
                spearman_df=spearman_df,
                university=school_key,
                feature_map={
                    "avg_s_jobs": "s_jobs",
                    "avg_s_level": "s_level",
                    "avg_s_prereqs": "s_prereqs",
                },
                normalize_score=True,
            )

            null_programme_job_matrix_df, _ = build_programme_job_matrix(
                modules_clean_df=null_module_feature_df,
                similarity_matrix=null_similarity_matrix,
                job_pool_df=job_pool_df,
                programme_col=programme_col,
                weight_col=weight_col,
                type_col=type_col,
                type_weight_map=type_weight_map,
                default_weight=default_weight,
                normalize_weights_within_programme=True,
                job_id_col="uuid",
                spearman_score_col=spearman_score_col,
            )

            null_score_chunks.append(null_programme_job_matrix_df.to_numpy(dtype=float).ravel())

        if (b + 1) % max(1, min(10, n_iter)) == 0 or (b + 1) == n_iter:
            print(f"Completed null iteration {b + 1}/{n_iter}")

    return np.concatenate(null_score_chunks, axis=0)


null_programme_job_scores = simulate_null_programme_job_scores(
    school_modules_dict=SCHOOL_MODULE_INPUTS,
    job_pool_df=job_pool_df,
    spearman_df=spearman_df,
    prereq_graphs_by_school=prereq_graphs,
    prereq_decay_by_school=prereq_decay_lookup,
    n_iter=NULL_N_ITER,
    random_state=42,
    module_embedding_col="skill_embedding",
    job_embedding_col="embedding_mpnet",
    programme_col="course",
    weight_col="curriculum_weight",
    type_col="type",
    type_weight_map=None,
    default_weight=1.0,
    spearman_score_col="spearman_module_score",
    top_k=TOP_K,
)

null_threshold_candidates = {
    "null_p95": float(np.quantile(null_programme_job_scores, 0.95)),
    "null_p97_5": float(np.quantile(null_programme_job_scores, 0.975)),
    "null_p99": float(np.quantile(null_programme_job_scores, 0.99)),
}

display(
    pd.DataFrame(
        {
            "candidate": list(null_threshold_candidates.keys()),
            "value": list(null_threshold_candidates.values()),
        }
    )
)

plt.figure(figsize=(10, 5.5))
plt.hist(
    real_programme_job_scores,
    bins=80,
    density=True,
    alpha=0.50,
    edgecolor="black",
    label="Real Pooled Programme-Job Scores",
)
plt.hist(
    null_programme_job_scores,
    bins=80,
    density=True,
    alpha=0.50,
    edgecolor="black",
    label="Null pooled programme-job scores",
)
for label, value in null_threshold_candidates.items():
    plt.axvline(value, linestyle="--", linewidth=1.3, label=f"{label} = {value:.3f}")
plt.title("Step 8B: Real vs Null Programme-Job Similarity Distributions", fontweight="bold")
plt.xlabel("Programme-Job Similarity")
plt.ylabel("Density")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
def build_threshold_result_df(
    threshold,
    job_pool_df,
    school_programme_matrices,
    school_module_feature_dfs,
    school_similarity_matrices,
    top_k=100,
    family_col="ssic_industry_family_2d",
):
    sweep_frames = []

    for school_name, programme_job_matrix_df in school_programme_matrices.items():
        df_school = build_programme_core_metrics(
            programme_job_matrix_df=programme_job_matrix_df,
            job_pool_df=job_pool_df,
            module_diagnostics_df=school_module_feature_dfs[school_name],
            similarity_matrix=school_similarity_matrices[school_name],
            similarity_threshold=threshold,
            top_k=top_k,
            programme_col="course",
            weight_col="curriculum_weight",
            type_col="type",
            type_weight_map=None,
            default_weight=1.0,
            family_col=family_col,
            spearman_score_col="spearman_module_score",
            school_name=school_name,
        )
        sweep_frames.append(df_school)

    out = pd.concat(sweep_frames, ignore_index=True)
    out = add_programme_display_label(out)
    out = add_percentile_rank_overview_score(
        out,
        metric_cols=[
            "reach_job_ratio",
            "mean_similarity_all_jobs",
            "top_k_mean_similarity",
            "max_similarity",
        ],
        new_col="semantic_overview_score",
    )
    out = assign_profile_labels_for_sweep(
        out,
        reach_col="reach_job_ratio",
        strength_col="top_k_mean_similarity",
        split_method="median",
    )
    out["threshold"] = threshold
    return out


def run_threshold_sweep(
    threshold_grid,
    job_pool_df,
    school_programme_matrices,
    school_module_feature_dfs,
    school_similarity_matrices,
    top_k=100,
    family_col="ssic_industry_family_2d",
    near_zero_cutoff=0.005,
    top_n_for_overlap=5,
    reference_threshold=0.35,
):
    threshold_results = {}
    summary_rows = []
    reference_df = None
    previous_top_set = None
    previous_profile_map = None

    for threshold in threshold_grid:
        df_t = build_threshold_result_df(
            threshold=threshold,
            job_pool_df=job_pool_df,
            school_programme_matrices=school_programme_matrices,
            school_module_feature_dfs=school_module_feature_dfs,
            school_similarity_matrices=school_similarity_matrices,
            top_k=top_k,
            family_col=family_col,
        )
        threshold_results[threshold] = df_t
        if np.isclose(threshold, reference_threshold):
            reference_df = df_t.copy()

    if reference_df is None:
        reference_threshold = float(threshold_grid[0])
        reference_df = threshold_results[reference_threshold].copy()

    reference_top_set = set(
        reference_df.sort_values("semantic_overview_score", ascending=False)
        .head(top_n_for_overlap)
        .apply(lambda r: f"{r['school']}|{r['course']}", axis=1)
        .tolist()
    )
    reference_profile_map = reference_df.set_index(["school", "course"])["opportunity_profile"].copy()

    for threshold in threshold_grid:
        df_t = threshold_results[threshold].copy()
        top_set = set(
            df_t.sort_values("semantic_overview_score", ascending=False)
            .head(top_n_for_overlap)
            .apply(lambda r: f"{r['school']}|{r['course']}", axis=1)
            .tolist()
        )
        profile_map = df_t.set_index(["school", "course"])["opportunity_profile"].copy()

        adjacent_top_overlap = np.nan
        adjacent_profile_stability = np.nan

        if previous_top_set is not None:
            adjacent_top_overlap = len(top_set & previous_top_set) / top_n_for_overlap

        if previous_profile_map is not None:
            aligned_prev = previous_profile_map.reindex(profile_map.index)
            adjacent_profile_stability = float((profile_map == aligned_prev).mean())

        reference_top_overlap = len(top_set & reference_top_set) / top_n_for_overlap
        aligned_ref = reference_profile_map.reindex(profile_map.index)
        reference_profile_stability = float((profile_map == aligned_ref).mean())

        summary_rows.append(
            {
                "threshold": threshold,
                "avg_reach_job_ratio": float(df_t["reach_job_ratio"].mean()),
                "median_reach_job_ratio": float(df_t["reach_job_ratio"].median()),
                "avg_matched_industry_family_count": float(df_t["matched_industry_family_count"].mean()),
                "median_matched_industry_family_count": float(df_t["matched_industry_family_count"].median()),
                "near_zero_fraction": float((df_t["reach_job_ratio"] <= near_zero_cutoff).mean()),
                "zero_reach_fraction": float((df_t["reach_job_ratio"] == 0).mean()),
                "adjacent_top_overlap": adjacent_top_overlap,
                "adjacent_profile_stability": adjacent_profile_stability,
                "reference_top_overlap": reference_top_overlap,
                "reference_profile_stability": reference_profile_stability,
            }
        )

        previous_top_set = top_set
        previous_profile_map = profile_map

    return threshold_results, pd.DataFrame(summary_rows), reference_threshold


threshold_results_dict, sweep_summary_df, sweep_reference_threshold = run_threshold_sweep(
    threshold_grid=THRESHOLD_GRID,
    job_pool_df=job_pool_df,
    school_programme_matrices=SCHOOL_PROGRAMME_MATRICES,
    school_module_feature_dfs=SCHOOL_MODULE_FEATURES,
    school_similarity_matrices=SCHOOL_SIMILARITY_MATRICES,
    top_k=TOP_K,
    family_col="ssic_industry_family_2d",
    near_zero_cutoff=0.005,
    top_n_for_overlap=5,
    reference_threshold=0.35,
)

display(sweep_summary_df)

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)

axes[0, 0].plot(
    sweep_summary_df["threshold"],
    sweep_summary_df["avg_reach_job_ratio"],
    marker="o",
)
axes[0, 0].set_title("Average Job Reach Ratio")
axes[0, 0].yaxis.set_major_formatter(PercentFormatter(xmax=1))
axes[0, 0].grid(alpha=0.25)

axes[0, 1].plot(
    sweep_summary_df["threshold"],
    sweep_summary_df["avg_matched_industry_family_count"],
    marker="o",
)
axes[0, 1].set_title("Average Matched 2-Digit Industry Family Count")
axes[0, 1].yaxis.set_major_locator(MaxNLocator(integer=True))
axes[0, 1].grid(alpha=0.25)

axes[1, 0].plot(
    sweep_summary_df["threshold"],
    sweep_summary_df["near_zero_fraction"],
    marker="o",
)
axes[1, 0].set_title("Proportion of Near-Zero Reach Programmes")
axes[1, 0].yaxis.set_major_formatter(PercentFormatter(xmax=1))
axes[1, 0].grid(alpha=0.25)

axes[1, 1].plot(
    sweep_summary_df["threshold"],
    sweep_summary_df["adjacent_top_overlap"],
    marker="o",
    label="Adjacent Top-5 Overlap",
)
axes[1, 1].plot(
    sweep_summary_df["threshold"],
    sweep_summary_df["adjacent_profile_stability"],
    marker="o",
    label="Adjacent Profile Stability",
)
axes[1, 1].set_title("Stability Across Neighbouring Thresholds")
axes[1, 1].set_ylim(0, 1.05)
axes[1, 1].yaxis.set_major_formatter(PercentFormatter(xmax=1))
axes[1, 1].grid(alpha=0.25)
axes[1, 1].legend()

for ax in axes[1, :]:
    ax.set_xlabel("Similarity Threshold")

fig.suptitle("Step 8C: Threshold Sweep Diagnostics", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
def build_tail_exceedance_df(real_scores, null_scores, threshold_grid):
    rows = []
    for t in threshold_grid:
        real_tail = float((real_scores >= t).mean())
        null_tail = float((null_scores >= t).mean())
        rows.append(
            {
                "threshold": float(t),
                "real_tail_fraction": real_tail,
                "null_tail_fraction": null_tail,
                "tail_enrichment_ratio": real_tail / max(null_tail, 1e-12),
            }
        )
    return pd.DataFrame(rows)


def enrich_sweep_summary_for_selection(sweep_summary_df):
    out = sweep_summary_df.copy()
    out["reach_retention"] = out["avg_reach_job_ratio"] / out["avg_reach_job_ratio"].max()
    out["breadth_retention"] = out["avg_matched_industry_family_count"] / out["avg_matched_industry_family_count"].max()
    return out


def choose_threshold_by_constraints(
    sweep_summary_df,
    null_threshold_candidates,
    null_floor_key="null_p99",
    min_adjacent_top_overlap=0.80,
    min_adjacent_profile_stability=0.80,
    max_near_zero_fraction=0.20,
    min_reach_retention=0.20,
    min_breadth_retention=0.75,
):
    df = enrich_sweep_summary_for_selection(sweep_summary_df)
    null_floor = float(null_threshold_candidates[null_floor_key])

    df["passes_null_floor"] = df["threshold"] >= null_floor
    df["passes_top_overlap"] = df["adjacent_top_overlap"].isna() | (df["adjacent_top_overlap"] >= min_adjacent_top_overlap)
    df["passes_profile_stability"] = df["adjacent_profile_stability"].isna() | (df["adjacent_profile_stability"] >= min_adjacent_profile_stability)
    df["passes_near_zero"] = df["near_zero_fraction"] <= max_near_zero_fraction
    df["passes_reach_retention"] = df["reach_retention"] >= min_reach_retention
    df["passes_breadth_retention"] = df["breadth_retention"] >= min_breadth_retention

    df["is_admissible"] = (
        df["passes_null_floor"]
        & df["passes_top_overlap"]
        & df["passes_profile_stability"]
        & df["passes_near_zero"]
        & df["passes_reach_retention"]
        & df["passes_breadth_retention"]
    )

    admissible_df = df[df["is_admissible"]].copy()
    chosen_threshold = float(admissible_df["threshold"].max()) if not admissible_df.empty else np.nan
    return df, admissible_df, chosen_threshold


def build_programme_job_long_with_metadata(programme_job_matrices, job_pool_df):
    long_frames = []
    for school_name, programme_job_matrix_df in programme_job_matrices.items():
        df_long = programme_job_matrix_df.copy().stack().reset_index()
        df_long.columns = ["course", "job_id", "programme_job_similarity"]
        df_long.insert(0, "school", school_name)
        long_frames.append(df_long)

    out = pd.concat(long_frames, ignore_index=True)
    job_meta = job_pool_df.copy()
    job_meta.columns = job_meta.columns.str.strip().str.lower()
    keep_cols = [col for col in ["uuid", "title", "description", "ssiccode", "ssic_industry_family_2d"] if col in job_meta.columns]
    job_meta = job_meta[keep_cols].copy()

    out = out.merge(job_meta, how="left", left_on="job_id", right_on="uuid")
    out = build_programme_label(out, school_col="school", course_col="course")

    if "description" in out.columns:
        out["description_snippet"] = (
            out["description"]
            .fillna("")
            .astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.slice(0, 180)
        )
    return out


def inspect_boundary_matches(programme_job_long_df, threshold, margin=0.02, n_each=10):
    df = programme_job_long_df.copy()
    above_df = (
        df[
            (df["programme_job_similarity"] >= threshold)
            & (df["programme_job_similarity"] <= threshold + margin)
        ]
        .sort_values("programme_job_similarity", ascending=True)
        .head(n_each)
        .copy()
    )
    below_df = (
        df[
            (df["programme_job_similarity"] < threshold)
            & (df["programme_job_similarity"] >= threshold - margin)
        ]
        .sort_values("programme_job_similarity", ascending=False)
        .head(n_each)
        .copy()
    )
    show_cols = [col for col in ["school", "programme_label", "job_id", "programme_job_similarity", "title", "ssiccode", "ssic_industry_family_2d", "description_snippet"] if col in df.columns]
    return above_df[show_cols], below_df[show_cols]


tail_exceedance_df = build_tail_exceedance_df(
    real_scores=real_programme_job_scores,
    null_scores=null_programme_job_scores,
    threshold_grid=THRESHOLD_GRID,
)

selection_df, admissible_thresholds_df, chosen_threshold = choose_threshold_by_constraints(
    sweep_summary_df=sweep_summary_df,
    null_threshold_candidates=null_threshold_candidates,
    null_floor_key="null_p99",
    min_adjacent_top_overlap=0.80,
    min_adjacent_profile_stability=0.80,
    max_near_zero_fraction=0.20,
    min_reach_retention=0.20,
    min_breadth_retention=0.75,
)

if pd.isna(chosen_threshold):
    raise ValueError("No admissible threshold found under the current constraints.")

SELECTED_SIMILARITY_THRESHOLD = float(chosen_threshold)

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)

axes[0, 0].plot(selection_df["threshold"], selection_df["reach_retention"], marker="o", label="Reach retention")
axes[0, 0].plot(selection_df["threshold"], selection_df["breadth_retention"], marker="o", label="Breadth retention")
axes[0, 0].axhline(0.20, color="gray", linestyle="--", linewidth=1.0)
axes[0, 0].axhline(0.75, color="gray", linestyle="--", linewidth=1.0)
axes[0, 0].axvline(SELECTED_SIMILARITY_THRESHOLD, color="black", linestyle="--", linewidth=1.2)
axes[0, 0].set_title("Retention of Reach and Breadth")
axes[0, 0].yaxis.set_major_formatter(PercentFormatter(xmax=1))
axes[0, 0].grid(alpha=0.25)
axes[0, 0].legend()

axes[0, 1].plot(selection_df["threshold"], selection_df["near_zero_fraction"], marker="o", label="Near-zero reach fraction")
axes[0, 1].axhline(0.20, color="gray", linestyle="--", linewidth=1.0)
axes[0, 1].axvline(SELECTED_SIMILARITY_THRESHOLD, color="black", linestyle="--", linewidth=1.2)
axes[0, 1].set_title("Collapse Diagnostic")
axes[0, 1].yaxis.set_major_formatter(PercentFormatter(xmax=1))
axes[0, 1].grid(alpha=0.25)
axes[0, 1].legend()

axes[1, 0].plot(selection_df["threshold"], selection_df["adjacent_top_overlap"], marker="o", label="Adjacent top-5 overlap")
axes[1, 0].plot(selection_df["threshold"], selection_df["adjacent_profile_stability"], marker="o", label="Adjacent profile stability")
axes[1, 0].axhline(0.80, color="gray", linestyle="--", linewidth=1.0)
axes[1, 0].axvline(SELECTED_SIMILARITY_THRESHOLD, color="black", linestyle="--", linewidth=1.2)
axes[1, 0].set_title("Stability Constraints")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].yaxis.set_major_formatter(PercentFormatter(xmax=1))
axes[1, 0].grid(alpha=0.25)
axes[1, 0].legend()

axes[1, 1].bar(
    selection_df["threshold"].astype(str),
    selection_df["is_admissible"].astype(int),
    color=np.where(selection_df["is_admissible"], "#59A14F", "#D3D3D3"),
    edgecolor="black",
)
axes[1, 1].bar(str(SELECTED_SIMILARITY_THRESHOLD), 1, color="#E15759", edgecolor="black")
axes[1, 1].set_title("Admissible Thresholds")
axes[1, 1].set_ylabel("Admissible (1 = Yes)")
axes[1, 1].set_ylim(0, 1.15)
axes[1, 1].yaxis.set_major_locator(MaxNLocator(integer=True))
axes[1, 1].grid(axis="y", alpha=0.25)

for ax in axes[1, :]:
    ax.set_xlabel("Similarity Threshold")

fig.suptitle("Step 8D: Threshold Selection Dashboard", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(tail_exceedance_df["threshold"], tail_exceedance_df["real_tail_fraction"], marker="o", label="Real tail fraction")
plt.plot(tail_exceedance_df["threshold"], tail_exceedance_df["null_tail_fraction"], marker="o", label="Null tail fraction")
plt.axvline(SELECTED_SIMILARITY_THRESHOLD, color="black", linestyle="--", linewidth=1.2, label=f"Chosen threshold = {SELECTED_SIMILARITY_THRESHOLD:.3f}")
plt.title("Real vs Null Tail Exceedance", fontweight="bold")
plt.xlabel("Similarity Threshold")
plt.ylabel("Fraction of Scores Above Threshold")
plt.gca().yaxis.set_major_formatter(PercentFormatter(xmax=1))
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

final_threshold_report_df = pd.DataFrame(
    {
        "item": [
            "Chosen universal threshold",
            "Null p95",
            "Null p97.5",
            "Null p99",
            "Admissible thresholds",
            "Industry source",
        ],
        "value": [
            f"{SELECTED_SIMILARITY_THRESHOLD:.3f}",
            f"{null_threshold_candidates['null_p95']:.4f}",
            f"{null_threshold_candidates['null_p97_5']:.4f}",
            f"{null_threshold_candidates['null_p99']:.4f}",
            ", ".join(f"{x:.3f}" for x in admissible_thresholds_df["threshold"].tolist()),
            job_pool_df.attrs.get("industry_source", "missing"),
        ],
    }
)
display(final_threshold_report_df)

all_programme_job_long_with_meta_df = build_programme_job_long_with_metadata(
    programme_job_matrices=SCHOOL_PROGRAMME_MATRICES,
    job_pool_df=job_pool_df,
)

boundary_checks = {}
for t in [0.25, SELECTED_SIMILARITY_THRESHOLD, 0.30]:
    above_df, below_df = inspect_boundary_matches(
        programme_job_long_df=all_programme_job_long_with_meta_df,
        threshold=t,
        margin=0.02,
        n_each=8,
    )
    boundary_checks[f"{t:.3f}_above"] = above_df
    boundary_checks[f"{t:.3f}_below"] = below_df

print(f"Selected similarity threshold: {SELECTED_SIMILARITY_THRESHOLD:.3f}")
print("Boundary sanity check examples")
for label, df in boundary_checks.items():
    print(label)
    display(df)


### Threshold Selection Conclusion

The selected threshold is applied only after the final Spearman-weighted programme-job matrices already exist. It is treated as a universal screening indicator because all programmes are evaluated on the same score scale against the same common job pool. The null comparison acts as a sanity floor, while the final choice is driven jointly by retention, stability, and qualitative boundary checks rather than by null separation alone.


# Part D — Results and Interpretation


## Step 9: Final Programme Results

This final section applies the selected threshold to the already-calibrated programme-job matrices and presents the main outputs in three views: semantic market alignment, breadth across industry families, and programme robustness.


### Result 1: Semantic Market Alignment

**Purpose:** summarize how strongly each programme aligns with the wider common job pool after threshold calibration is complete.

**What is shown:** school-level dual-axis panels for reach and mean similarity, a normalized heatmap across the main semantic metrics, and a cross-school ranked table of the top 10 programmes by post-threshold job reach ratio.

**Why it comes first:** this is the broad external alignment view, so it provides the baseline interpretation before moving to industry breadth and internal curriculum robustness.


In [ ]:
final_core_metric_frames = []
for school_name in [SCHOOL_DISPLAY[school] for school in SCHOOLS]:
    final_core_metric_frames.append(
        build_programme_core_metrics(
            programme_job_matrix_df=SCHOOL_PROGRAMME_MATRICES[school_name],
            job_pool_df=job_pool_df,
            module_diagnostics_df=SCHOOL_MODULE_FEATURES[school_name],
            similarity_matrix=SCHOOL_SIMILARITY_MATRICES[school_name],
            similarity_threshold=SELECTED_SIMILARITY_THRESHOLD,
            top_k=TOP_K,
            programme_col="course",
            weight_col="curriculum_weight",
            type_col="type",
            type_weight_map=None,
            default_weight=1.0,
            family_col="ssic_industry_family_2d",
            spearman_score_col="spearman_module_score",
            school_name=school_name,
        )
    )

result1_df = pd.concat(final_core_metric_frames, ignore_index=True).copy()
result1_df = result1_df.merge(
    pd.concat(
        [
            school_programme_metadata[school_name.lower()].assign(school=school_name)
            for school_name in [SCHOOL_DISPLAY[school] for school in SCHOOLS]
        ],
        ignore_index=True,
    ),
    how="left",
    on=["school", "course"],
)
result1_df = add_programme_display_label(result1_df)
result1_df = add_percentile_rank_overview_score(
    result1_df,
    metric_cols=[
        "reach_job_ratio",
        "mean_similarity_all_jobs",
        "top_k_mean_similarity",
        "max_similarity",
    ],
    new_col="semantic_overview_score",
)

print("Result 1 dataframe shape:", result1_df.shape)
display(result1_df.head())


def plot_result1a_dual_axis_by_school(
    df,
    reach_col="reach_job_ratio",
    mean_col="mean_similarity_all_jobs",
    sort_cols=None,
    school_order=None,
    percent_decimals=2,
    figsize=(21, 8),
):
    plot_df = df.copy()
    if school_order is None:
        school_order = [SCHOOL_DISPLAY[school] for school in SCHOOLS]
    if sort_cols is None:
        sort_cols = [reach_col, mean_col]

    fig, axes = plt.subplots(1, len(school_order), figsize=figsize, sharey=False)
    if len(school_order) == 1:
        axes = [axes]

    for ax, school_name in zip(axes, school_order):
        df_school = plot_df[plot_df["school"].astype(str).str.upper() == school_name.upper()].copy()
        df_school = df_school.sort_values(by=sort_cols, ascending=[False] * len(sort_cols)).reset_index(drop=True)
        x = np.arange(len(df_school))

        school_key = school_name.lower()
        bar_color = school_colors[school_key]["primary"]
        line_color = school_colors[school_key]["secondary"]

        bars = ax.bar(x, df_school[reach_col], color=bar_color, edgecolor="black", alpha=0.9)
        ax.set_title(f"{school_name}", fontweight="bold")
        ax.set_xlabel("Programme")
        ax.set_ylabel("Job Reach Ratio")
        ax.set_xticks(x)
        ax.set_xticklabels(df_school["programme_only_label"], rotation=60, ha="right")
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=2))
        ax.grid(axis="y", linestyle="--", alpha=0.25)

        ax2 = ax.twinx()
        line, = ax2.plot(
            x,
            df_school[mean_col],
            color=line_color,
            marker="o",
            linewidth=2.2,
            markersize=6,
        )
        ax2.set_ylabel("Mean Similarity Across All Jobs")

        reach_values = df_school[reach_col].to_numpy(dtype=float)
        reach_pad = max(np.nanmax(reach_values) * 0.015, 0.0002)
        for bar, value in zip(bars, reach_values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                value + reach_pad,
                f"{value:.{percent_decimals}%}",
                ha="center",
                va="bottom",
                fontsize=8,
            )

        mean_values = df_school[mean_col].to_numpy(dtype=float)
        mean_pad = max(np.nanmax(mean_values) * 0.01, 0.001)
        for xi, value in zip(x, mean_values):
            ax2.text(xi, value + mean_pad, f"{value:.3f}", ha="center", va="bottom", fontsize=8, color=line_color)

        ax.legend([bars, line], ["Job Reach Ratio", "Mean Similarity"], loc="upper right", frameon=True)

    fig.suptitle("Result 1A: Semantic Market Alignment by School", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()


def plot_result1b_alignment_heatmap(
    df,
    metrics_cols=None,
    sort_by="semantic_overview_score",
    figsize=(10, 14),
    annotate=True,
):
    if metrics_cols is None:
        metrics_cols = [
            "reach_job_ratio",
            "mean_similarity_all_jobs",
            "top_k_mean_similarity",
            "max_similarity",
            "semantic_overview_score",
        ]

    metric_label_map = {
        "reach_job_ratio": "Reach ratio",
        "mean_similarity_all_jobs": "Mean similarity",
        "top_k_mean_similarity": "Top-k mean similarity",
        "max_similarity": "Max similarity",
        "semantic_overview_score": "Rank-convenience score",
    }

    plot_df = df.copy().sort_values(sort_by, ascending=False).reset_index(drop=True)
    raw_heatmap_data = plot_df.set_index("programme_label")[metrics_cols].copy()
    norm_heatmap_data = minmax_normalize_columns(raw_heatmap_data, metrics_cols)

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(norm_heatmap_data.to_numpy(dtype=float), aspect="auto", cmap="YlOrRd")

    ax.set_title("Result 1B: Semantic Market Alignment Heatmap", fontweight="bold")
    ax.set_xticks(np.arange(len(metrics_cols)))
    ax.set_xticklabels([metric_label_map.get(col, col) for col in metrics_cols], rotation=30, ha="right")
    ax.set_yticks(np.arange(len(raw_heatmap_data.index)))
    ax.set_yticklabels(raw_heatmap_data.index)

    if annotate:
        raw_values = raw_heatmap_data.to_numpy(dtype=float)
        for i in range(raw_values.shape[0]):
            for j, col in enumerate(metrics_cols):
                value = raw_values[i, j]
                text_label = f"{value:.2%}" if col == "reach_job_ratio" else f"{value:.3f}"
                ax.text(j, i, text_label, ha="center", va="center", fontsize=7, color="black")

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Column-Normalized Value")
    plt.tight_layout()
    plt.show()


def build_result1_top10_reach_table(df, top_n=10):
    out = df[
        [
            "school",
            "course",
            "programme_only_label",
            "reach_job_ratio",
            "top_k_mean_similarity",
            "mean_similarity_all_jobs",
            "max_similarity",
        ]
    ].copy()

    out["course"] = out["programme_only_label"].fillna(out["course"]).astype(str).str.strip()
    out = out.drop(columns=["programme_only_label"])
    out = out.sort_values(
        ["reach_job_ratio", "top_k_mean_similarity", "mean_similarity_all_jobs"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    out.insert(0, "rank", np.arange(1, len(out) + 1))
    out = out.head(top_n).copy()

    out["job reach ratio"] = pd.to_numeric(out["reach_job_ratio"], errors="coerce").map(
        lambda x: f"{x:.1%}" if pd.notna(x) else ""
    )
    for col in ["top_k_mean_similarity", "mean_similarity_all_jobs", "max_similarity"]:
        out[col] = pd.to_numeric(out[col], errors="coerce").round(3)

    return out[
        [
            "rank",
            "school",
            "course",
            "job reach ratio",
            "top_k_mean_similarity",
            "mean_similarity_all_jobs",
            "max_similarity",
        ]
    ]


plot_result1a_dual_axis_by_school(result1_df)
plot_result1b_alignment_heatmap(result1_df)

print("The table below ranks the programmes with the highest post-threshold labour-market reach across NUS, SMU, and SUTD.")
result1_top10_reach_table = build_result1_top10_reach_table(result1_df, top_n=10)
display(result1_top10_reach_table)


### Result 2: Breadth Across Industry Families

**Purpose:** examine whether programme reach is spread across many reachable 2-digit industry families or concentrated within a smaller set of industries.

**What is shown:** a thresholded pie chart of the most common reachable 2-digit SSIC industry families, a breadth-versus-reach scatter, and a heatmap of the broadest programmes by industry composition.

**Interpretation:** SSIC reflects the employer's industry rather than the occupation itself, so this section describes industry spread and industry composition, not occupational breadth.


In [ ]:
def build_programme_industry_composition_long(
    programme_job_matrix_df,
    job_pool_df,
    similarity_threshold,
    industry_family_col="ssic_industry_family_2d",
    school_name=None,
):
    if programme_job_matrix_df.empty:
        return pd.DataFrame()

    matrix = programme_job_matrix_df.to_numpy(dtype=float)
    programmes = programme_job_matrix_df.index.astype(str).tolist()
    industry_family_values = job_pool_df[industry_family_col].fillna("").astype(str).to_numpy()
    valid_industry_family_mask = np.array([bool(x) and x.lower() != "nan" for x in industry_family_values], dtype=bool)
    reach_mask = matrix >= similarity_threshold

    long_rows = []
    for i, programme in enumerate(programmes):
        reached_idx = np.flatnonzero(reach_mask[i] & valid_industry_family_mask)
        family_counts = (
            pd.Series(industry_family_values[reached_idx]).value_counts()
            if reached_idx.size > 0
            else pd.Series(dtype=int)
        )
        total_reached_jobs = int(reached_idx.size)

        for industry_family_code, count_val in family_counts.items():
            long_rows.append(
                {
                    "course": programme,
                    "industry_family_code": industry_family_code,
                    "count": int(count_val),
                    "proportion": (int(count_val) / total_reached_jobs) if total_reached_jobs > 0 else 0.0,
                }
            )

    long_df = pd.DataFrame(long_rows)
    if school_name is not None and not long_df.empty:
        long_df.insert(0, "school", school_name)
    return long_df


result2_df = result1_df[
    [
        "school",
        "course",
        "programme_only_label",
        "programme_display_label",
        "reach_job_ratio",
        "matched_industry_family_count",
        "matched_industry_family_share",
        "normalized_industry_family_entropy",
        "top_k_mean_similarity",
        "semantic_overview_score",
    ]
].copy()

industry_long_frames = []
for school_name in [SCHOOL_DISPLAY[school] for school in SCHOOLS]:
    industry_long_frames.append(
        build_programme_industry_composition_long(
            programme_job_matrix_df=SCHOOL_PROGRAMME_MATRICES[school_name],
            job_pool_df=job_pool_df,
            similarity_threshold=SELECTED_SIMILARITY_THRESHOLD,
            industry_family_col="ssic_industry_family_2d",
            school_name=school_name,
        )
    )
all_programme_industry_long_df = (
    pd.concat(industry_long_frames, ignore_index=True)
    if industry_long_frames
    else pd.DataFrame()
)


def _compute_scatter_label_offsets(df, x_col, y_col, y_threshold=0.015):
    offsets = {}
    for x_val, group in df.groupby(x_col):
        group = group.sort_values(y_col).copy()
        cluster, prev_y = [], None

        def assign_cluster_offsets(cluster_rows):
            n = len(cluster_rows)
            if n == 1:
                offsets[cluster_rows[0]] = (6, 4)
                return
            y_offsets = np.linspace(-(n - 1) / 2, (n - 1) / 2, n) * 10
            for idx, yo in zip(cluster_rows, y_offsets):
                offsets[idx] = (6, float(yo))

        for idx, row in group.iterrows():
            y_val = float(row[y_col])
            if prev_y is None or abs(y_val - prev_y) <= y_threshold:
                cluster.append(idx)
            else:
                assign_cluster_offsets(cluster)
                cluster = [idx]
            prev_y = y_val

        if cluster:
            assign_cluster_offsets(cluster)

    return offsets


# SSIC 2-digit industry labels used for readable Result 2 plots.
# Unmapped codes remain explicit rather than guessed.
SSIC_2D_LABEL_MAP = {
    "01": "Agriculture, Hunting and Related Service Activities",
    "02": "Forestry, Logging and Related Service Activities",
    "03": "Fishing, Aquaculture and Service Activities Incidental to Fishing",
    "05": "Mining of Coal and Lignite",
    "06": "Extraction of Crude Petroleum and Natural Gas",
    "07": "Mining of Metal Ores",
    "08": "Other Mining and Quarrying",
    "09": "Mining Support Service Activities",
    "10": "Manufacture of Food Products",
    "11": "Manufacture of Beverages",
    "12": "Manufacture of Tobacco Products",
    "13": "Manufacture of Textiles",
    "14": "Manufacture of Wearing Apparel",
    "15": "Manufacture of Leather and Related Products",
    "16": "Manufacture of Wood and of Products of Wood and Cork, Except Furniture",
    "17": "Manufacture of Paper and Paper Products",
    "18": "Printing and Reproduction of Recorded Media",
    "19": "Manufacture of Coke and Refined Petroleum Products",
    "20": "Manufacture of Chemicals and Chemical Products",
    "21": "Manufacture of Pharmaceuticals, Medicinal Chemical and Botanical Products",
    "22": "Manufacture of Rubber and Plastic Products",
    "23": "Manufacture of Other Non-Metallic Mineral Products",
    "24": "Manufacture of Basic Metals",
    "25": "Manufacture of Fabricated Metal Products, Except Machinery and Equipment",
    "26": "Manufacture of Computer, Electronic and Optical Products",
    "27": "Manufacture of Electrical Equipment",
    "28": "Manufacture of Machinery and Equipment n.e.c.",
    "29": "Manufacture of Motor Vehicles, Trailers and Semi-Trailers",
    "30": "Manufacture of Other Transport Equipment",
    "31": "Manufacture of Furniture",
    "32": "Other Manufacturing",
    "33": "Repair and Installation of Machinery and Equipment",
    "35": "Electricity, Gas, Steam and Air Conditioning Supply",
    "36": "Water Collection, Treatment and Supply",
    "37": "Sewerage",
    "38": "Waste Collection, Treatment and Disposal Activities; Materials Recovery",
    "39": "Remediation Activities and Other Waste Management Services",
    "41": "Construction of Buildings",
    "42": "Civil Engineering",
    "43": "Specialised Construction Activities",
    "45": "Wholesale and Retail Trade and Repair of Motor Vehicles and Motorcycles",
    "46": "Wholesale Trade, Except of Motor Vehicles and Motorcycles",
    "47": "Retail Trade, Except of Motor Vehicles and Motorcycles",
    "49": "Land Transport and Transport via Pipelines",
    "50": "Water Transport",
    "51": "Air Transport",
    "52": "Warehousing and Support Activities for Transportation",
    "53": "Postal and Courier Activities",
    "55": "Accommodation",
    "56": "Food and Beverage Service Activities",
    "58": "Publishing Activities",
    "59": "Motion Picture, Video and Television Programme Production, Sound Recording and Music Publishing Activities",
    "60": "Programming and Broadcasting Activities",
    "61": "Telecommunications",
    "62": "Computer Programming, Consultancy and Related Activities",
    "63": "Information Service Activities",
    "64": "Financial Service Activities, Except Insurance and Pension Funding",
    "65": "Insurance, Reinsurance and Pension Funding, Except Compulsory Social Security",
    "66": "Activities Auxiliary to Financial Service and Insurance Activities",
    "68": "Real Estate Activities",
    "69": "Legal and Accounting Activities",
    "70": "Activities of Head Offices; Management Consultancy Activities",
    "71": "Architectural and Engineering Activities; Technical Testing and Analysis",
    "72": "Scientific Research and Development",
    "73": "Advertising and Market Research",
    "74": "Other Professional, Scientific and Technical Activities",
    "75": "Veterinary Activities",
    "77": "Rental and Leasing Activities",
    "78": "Employment Activities",
    "79": "Travel Agency, Tour Operator and Other Reservation Service and Related Activities",
    "80": "Security and Investigation Activities",
    "81": "Services to Buildings and Landscape Activities",
    "82": "Office Administrative, Office Support and Other Business Support Activities",
    "84": "Public Administration and Defence; Compulsory Social Security",
    "85": "Education",
    "86": "Human Health Activities",
    "87": "Residential Care Activities",
    "88": "Social Work Activities Without Accommodation",
    "90": "Creative, Arts and Entertainment Activities",
    "91": "Libraries, Archives, Museums and Other Cultural Activities",
    "92": "Gambling and Betting Activities",
    "93": "Sports Activities and Amusement and Recreation Activities",
    "94": "Activities of Membership Organisations",
    "95": "Repair of Computers and Personal and Household Goods",
    "96": "Other Personal Service Activities",
    "97": "Activities of Households as Employers of Domestic Personnel",
    "98": "Undifferentiated Goods- and Services-Producing Activities of Private Households for Own Use",
    "99": "Activities of Extraterritorial Organisations and Bodies",
}


def get_industry_display_label(industry_family_code, industry_label_map=None):
    industry_family_code = str(industry_family_code).strip()
    industry_label_map = industry_label_map or SSIC_2D_LABEL_MAP
    industry_name = industry_label_map.get(industry_family_code)

    if industry_name:
        return f"{industry_family_code} — {industry_name}"
    return f"{industry_family_code} — Unmapped SSIC Industry"


def build_reachable_industry_family_summary(
    programme_job_long_df,
    job_pool_df,
    similarity_threshold,
    industry_family_col="ssic_industry_family_2d",
    top_n=10,
):
    programme_job_long_df = programme_job_long_df.copy()
    job_pool = job_pool_df.copy()
    job_pool.columns = job_pool.columns.str.strip().str.lower()

    reachable_job_ids = (
        programme_job_long_df.loc[
            programme_job_long_df["programme_job_similarity"] >= similarity_threshold,
            "job_id",
        ]
        .dropna()
        .astype(str)
        .unique()
    )

    reachable_jobs = job_pool[job_pool["uuid"].astype(str).isin(reachable_job_ids)].copy()
    industry_family_series = reachable_jobs[industry_family_col].dropna().astype(str)
    industry_family_series = industry_family_series.loc[industry_family_series.str.strip() != ""]

    industry_family_counts = (
        industry_family_series.value_counts()
        .rename_axis("industry_family_code")
        .reset_index(name="reachable_job_count")
        .sort_values("reachable_job_count", ascending=False)
        .reset_index(drop=True)
    )

    industry_family_counts["industry_family_label"] = industry_family_counts["industry_family_code"].map(
        lambda code: get_industry_display_label(code)
    )

    if len(industry_family_counts) > top_n:
        other_count = int(industry_family_counts.iloc[top_n:]["reachable_job_count"].sum())
        top_df = industry_family_counts.head(top_n).copy()
        if other_count > 0:
            top_df = pd.concat(
                [
                    top_df,
                    pd.DataFrame(
                        {
                            "industry_family_code": ["Other"],
                            "reachable_job_count": [other_count],
                            "industry_family_label": ["Other — Remaining Reachable Industry Families"],
                        }
                    ),
                ],
                ignore_index=True,
            )
        industry_family_counts = top_df

    industry_family_counts["share_of_reachable_jobs"] = (
        industry_family_counts["reachable_job_count"] / industry_family_counts["reachable_job_count"].sum()
    )
    return industry_family_counts


def plot_result2a_reachable_industry_pie(summary_df, figsize=(12, 8)):
    plot_df = summary_df.copy()
    colors = list(plt.cm.tab20.colors[: len(plot_df)])

    fig, ax = plt.subplots(figsize=figsize)
    wedges, _ = ax.pie(
        plot_df["reachable_job_count"],
        labels=None,
        colors=colors,
        startangle=90,
        counterclock=False,
        wedgeprops=dict(edgecolor="white", linewidth=1),
    )

    total = plot_df["reachable_job_count"].sum()

    for wedge, _, row in zip(wedges, colors, plot_df.itertuples(index=False)):
        if row.industry_family_code == "Other":
            label_text = f"Other\n{row.share_of_reachable_jobs:.1%}"
        else:
            label_text = f"{row.industry_family_code}\n{row.share_of_reachable_jobs:.1%}"

        angle = (wedge.theta2 + wedge.theta1) / 2.0
        x = np.cos(np.deg2rad(angle))
        y = np.sin(np.deg2rad(angle))

        ax.annotate(
            label_text,
            xy=(0.92 * x, 0.92 * y),
            xytext=(1.25 * np.sign(x), 1.15 * y),
            ha="left" if x >= 0 else "right",
            va="center",
            fontsize=9,
            arrowprops=dict(arrowstyle="-", color="gray", lw=0.8),
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.8),
        )

    legend_labels = [
        f"{label} ({count:,})"
        for label, count in zip(
            plot_df["industry_family_label"],
            plot_df["reachable_job_count"],
        )
    ]

    ax.legend(
        wedges,
        legend_labels,
        title="Reachable 2-Digit Industry Family",
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        frameon=True,
    )
    ax.set_title("Result 2A: Most Common Reachable 2-Digit Industry Families", fontweight="bold")
    plt.tight_layout()
    plt.show()


def plot_result2b_industry_breadth_vs_reach_scatter(
    df,
    x_col="matched_industry_family_count",
    y_col="reach_job_ratio",
    figsize=(12, 8),
):
    plot_df = df.copy().reset_index(drop=True)
    offsets = _compute_scatter_label_offsets(plot_df, x_col=x_col, y_col=y_col, y_threshold=0.015)

    fig, ax = plt.subplots(figsize=figsize)
    for idx, row in plot_df.iterrows():
        school_key = str(row["school"]).lower()
        x_val = float(row[x_col])
        y_val = float(row[y_col])
        ax.scatter(
            x_val,
            y_val,
            s=130,
            color=school_colors[school_key]["primary"],
            edgecolor="black",
            alpha=0.9,
            zorder=3,
        )
        x_off, y_off = offsets.get(idx, (6, 4))
        ax.annotate(
            row["programme_display_label"],
            xy=(x_val, y_val),
            xytext=(x_off, y_off),
            textcoords="offset points",
            fontsize=9,
            arrowprops=dict(arrowstyle="-", color="gray", lw=0.8, shrinkA=0, shrinkB=4),
        )

    ax.set_title("Result 2B: Industry Breadth Versus Job Reach", fontweight="bold")
    ax.set_xlabel("Matched 2-Digit Industry Family Count")
    ax.set_ylabel("Job Reach Ratio")
    ax.grid(alpha=0.25)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))

    school_legend_handles = [
        Line2D(
            [0], [0], marker="o", color="w", label=school.upper(),
            markerfacecolor=school_colors[school]["primary"],
            markeredgecolor="black", markersize=10,
        )
        for school in ["nus", "smu", "sutd"]
    ]
    ax.legend(handles=school_legend_handles, title="University", loc="lower right", frameon=True)
    plt.tight_layout()
    plt.show()


def plot_result2c_top_broad_programmes_heatmap(
    result2_df,
    long_df,
    top_n_programmes=6,
    top_n_families=12,
    sort_cols=None,
    figsize=(14, 7),
    annotate=True,
):
    if sort_cols is None:
        sort_cols = [
            "matched_industry_family_count",
            "normalized_industry_family_entropy",
            "reach_job_ratio",
        ]

    top_df = (
        result2_df.sort_values(sort_cols, ascending=[False] * len(sort_cols))
        .head(top_n_programmes)
        .copy()
    )
    selected_keys = top_df[["school", "course"]].drop_duplicates()
    heatmap_df = long_df.merge(selected_keys, on=["school", "course"], how="inner").copy()

    if heatmap_df.empty:
        print("No industry-composition rows available for the selected programmes.")
        return top_df

    top_families = (
        heatmap_df.groupby("industry_family_code")["count"]
        .sum()
        .sort_values(ascending=False)
        .head(top_n_families)
        .index
        .tolist()
    )

    heatmap_df = heatmap_df[heatmap_df["industry_family_code"].isin(top_families)].copy()
    row_meta = top_df[
        ["school", "course", "programme_display_label", "matched_industry_family_count"]
    ].drop_duplicates().copy()

    row_meta["row_label"] = (
        row_meta["programme_display_label"]
        + "\n"
        + row_meta["matched_industry_family_count"].astype(int).astype(str)
        + " industry families reached"
    )

    heatmap_df = heatmap_df.merge(
        row_meta[["school", "course", "row_label"]],
        on=["school", "course"],
        how="left",
    )

    heatmap_matrix = (
        heatmap_df.pivot_table(
            index="row_label",
            columns="industry_family_code",
            values="proportion",
            aggfunc="sum",
            fill_value=0,
        )
        .reindex(index=row_meta["row_label"], columns=top_families, fill_value=0)
    )

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(
        heatmap_matrix.to_numpy(dtype=float),
        aspect="auto",
        cmap="YlGnBu",
        vmin=0,
        vmax=max(0.01, float(heatmap_matrix.to_numpy(dtype=float).max())),
    )
    ax.set_title("Result 2C: Composition of the Broadest Programmes by 2-Digit Industry Family", fontweight="bold")
    ax.set_xticks(np.arange(len(heatmap_matrix.columns)))
    ax.set_xticklabels(heatmap_matrix.columns, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(heatmap_matrix.index)))
    ax.set_yticklabels(heatmap_matrix.index)

    if annotate:
        values = heatmap_matrix.to_numpy(dtype=float)
        for i in range(values.shape[0]):
            for j in range(values.shape[1]):
                ax.text(
                    j, i, f"{values[i, j]:.1%}",
                    ha="center", va="center",
                    fontsize=8, color="black"
                )

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Proportion of Reached Jobs")
    plt.tight_layout()
    plt.show()
    return top_df


print(
    "Result 2A shows the composition of jobs that are reachable by at least one programme "
    "under the selected threshold. It does not show the raw composition of the original job pool."
)

result2a_reachable_industry_summary_df = build_reachable_industry_family_summary(
    programme_job_long_df=all_programme_job_long_df,
    job_pool_df=job_pool_df,
    similarity_threshold=SELECTED_SIMILARITY_THRESHOLD,
    industry_family_col="ssic_industry_family_2d",
    top_n=10,
)

plot_result2a_reachable_industry_pie(result2a_reachable_industry_summary_df)
plot_result2b_industry_breadth_vs_reach_scatter(result2_df)

top_broad_programmes_df = plot_result2c_top_broad_programmes_heatmap(
    result2_df=result2_df,
    long_df=all_programme_industry_long_df,
    top_n_programmes=6,
    top_n_families=12,
)

print("Result 2 DataFrame")
display(result2_df.head())

print("Top Broad Programmes Used in Result 2C")
display(top_broad_programmes_df)

### Result 3: Programme Robustness

**Purpose:** assess whether observed employability breadth is supported broadly across the curriculum or depends heavily on a small number of modules.

**What is shown:** stacked robustness breakdowns for concentrated and distributed programmes, plus boxplots of within-programme module contribution shares.

**Why it comes after the external results:** robustness is an internal curriculum diagnostic, so it is most informative once the external employability patterns are already visible.


In [ ]:
def prepare_result4_df(result1_df):
    required_cols = [
        "school",
        "course",
        "programme_display_label",
        "programme_only_label",
        "top1_module_share",
        "top5_module_share",
        "rest_module_share",
        "effective_module_count",
        "normalized_effective_module_count",
        "reach_job_ratio",
        "top_k_mean_similarity",
        "semantic_overview_score",
    ]
    missing_cols = [col for col in required_cols if col not in result1_df.columns]
    if missing_cols:
        raise KeyError(f"Missing required column(s) in result1_df: {missing_cols}")
    return result1_df[required_cols].copy()


result4_df = prepare_result4_df(result1_df)


def select_result4_extremes(result4_df, top_n=5):
    df = result4_df.copy()
    concentrated_df = (
        df.sort_values(
            ["top1_module_share", "top5_module_share", "normalized_effective_module_count"],
            ascending=[False, False, True],
        )
        .head(top_n)
        .copy()
    )

    concentrated_keys = set(zip(concentrated_df["school"], concentrated_df["course"]))
    remaining_df = df.loc[
        ~df.apply(lambda row: (row["school"], row["course"]) in concentrated_keys, axis=1)
    ].copy()

    distributed_df = (
        remaining_df.sort_values(
            ["normalized_effective_module_count", "top1_module_share", "top5_module_share"],
            ascending=[False, True, True],
        )
        .head(top_n)
        .copy()
    )
    return concentrated_df, distributed_df


def build_module_contribution_share_long_df(
    module_diagnostics_df,
    similarity_matrix,
    programme_job_matrix_df,
    programme_col="course",
    weight_col="curriculum_weight",
    type_col="type",
    type_weight_map=None,
    default_weight=1.0,
    spearman_score_col="spearman_module_score",
    top_k_jobs=100,
    school_name=None,
):
    modules = module_diagnostics_df.copy()
    modules.columns = modules.columns.str.strip().str.lower()
    programme_col = programme_col.lower()
    modules[programme_col] = modules[programme_col].fillna("").astype(str).str.strip().str.lower()
    modules = modules[modules[programme_col] != ""].copy()
    modules["_module_idx"] = modules.get("_module_idx", np.arange(len(modules)))

    modules["_effective_weight"] = _resolve_module_weights(
        modules_df=modules,
        weight_col=weight_col,
        type_col=type_col,
        type_weight_map=type_weight_map,
        default_weight=default_weight,
        spearman_score_col=spearman_score_col,
    )

    output_rows = []
    for programme in programme_job_matrix_df.index.astype(str):
        programme_key = str(programme).strip().lower()
        group = modules[modules[programme_col] == programme_key].copy()
        if group.empty:
            continue

        programme_vector = programme_job_matrix_df.loc[programme].to_numpy(dtype=float)
        n_jobs = programme_vector.shape[0]
        k = min(max(int(top_k_jobs), 1), n_jobs)
        top_job_idx = np.arange(n_jobs) if k == n_jobs else np.sort(np.argpartition(programme_vector, n_jobs - k)[-k:])

        module_row_idx = group["_module_idx"].to_numpy(dtype=int)
        module_weights = group["_effective_weight"].to_numpy(dtype=float)
        module_strength = similarity_matrix[np.ix_(module_row_idx, top_job_idx)].mean(axis=1)
        weighted_contribution = module_weights * np.clip(module_strength, a_min=0, a_max=None)
        contribution_shares = weighted_contribution / weighted_contribution.sum() if weighted_contribution.sum() > 0 else module_weights / module_weights.sum()

        group["module_contribution_share"] = contribution_shares
        group = group.sort_values("module_contribution_share", ascending=False).reset_index(drop=True)
        group["module_rank"] = np.arange(1, len(group) + 1)

        for _, row in group.iterrows():
            output_rows.append(
                {
                    "school": school_name,
                    "course": programme_key,
                    "module_code": row.get("code", ""),
                    "module_type": row.get("type", ""),
                    "module_rank": int(row["module_rank"]),
                    "module_contribution_share": float(row["module_contribution_share"]),
                }
            )

    return pd.DataFrame(output_rows)


module_contrib_frames = []
for school_name in [SCHOOL_DISPLAY[school] for school in SCHOOLS]:
    module_contrib_frames.append(
        build_module_contribution_share_long_df(
            module_diagnostics_df=SCHOOL_MODULE_FEATURES[school_name],
            similarity_matrix=SCHOOL_SIMILARITY_MATRICES[school_name],
            programme_job_matrix_df=SCHOOL_PROGRAMME_MATRICES[school_name],
            programme_col="course",
            weight_col="curriculum_weight",
            type_col="type",
            type_weight_map=None,
            default_weight=1.0,
            spearman_score_col="spearman_module_score",
            top_k_jobs=TOP_K,
            school_name=school_name,
        )
    )

all_module_contrib_long_df = pd.concat(module_contrib_frames, ignore_index=True)
all_module_contrib_long_df = all_module_contrib_long_df.merge(
    result4_df[["school", "course", "programme_display_label"]].drop_duplicates(),
    how="left",
    on=["school", "course"],
)


def plot_result4a_robustness_breakdown(result4_df, top_n=5, figsize=(18, 7)):
    concentrated_df, distributed_df = select_result4_extremes(result4_df=result4_df, top_n=top_n)
    for df_sel in [concentrated_df, distributed_df]:
        df_sel["top5_minus_top1_share"] = df_sel["top5_module_share"] - df_sel["top1_module_share"]

    fig, axes = plt.subplots(1, 2, figsize=figsize, sharex=True)
    panel_specs = [
        (axes[0], concentrated_df.sort_values("top1_module_share", ascending=True), "Most concentrated programmes"),
        (axes[1], distributed_df.sort_values("normalized_effective_module_count", ascending=True), "Most distributed programmes"),
    ]

    for ax, df_plot, title in panel_specs:
        labels = df_plot["programme_display_label"].tolist()
        y = np.arange(len(df_plot))
        school_primary = [school_colors[s.lower()]["primary"] for s in df_plot["school"]]
        school_secondary = [school_colors[s.lower()]["secondary"] for s in df_plot["school"]]

        ax.barh(y, df_plot["top1_module_share"], color=school_primary, edgecolor="black", label="Top 1 module")
        ax.barh(y, df_plot["top5_minus_top1_share"], left=df_plot["top1_module_share"], color=school_secondary, edgecolor="black", alpha=0.9, label="Top 5 minus Top 1")
        ax.barh(y, df_plot["rest_module_share"], left=df_plot["top5_module_share"], color="#D3D3D3", edgecolor="black", label="Remaining modules")
        ax.set_yticks(y)
        ax.set_yticklabels(labels)
        ax.set_title(title, fontweight="bold")
        ax.xaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
        ax.grid(axis="x", alpha=0.25)

    axes[0].set_xlabel("Share of Total Programme Contribution")
    axes[1].set_xlabel("Share of Total Programme Contribution")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3, frameon=True, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle("Result 3A: Programme Robustness Breakdown", fontsize=16, fontweight="bold", y=1.06)
    plt.tight_layout()
    plt.show()


def plot_result4b_module_contribution_boxplots(result4_df, module_contrib_long_df, top_n=5, figsize=(18, 7)):
    concentrated_df, distributed_df = select_result4_extremes(result4_df=result4_df, top_n=top_n)
    concentrated_keys = concentrated_df[["school", "course"]].drop_duplicates()
    distributed_keys = distributed_df[["school", "course"]].drop_duplicates()

    left_df = module_contrib_long_df.merge(concentrated_keys, on=["school", "course"], how="inner").copy()
    right_df = module_contrib_long_df.merge(distributed_keys, on=["school", "course"], how="inner").copy()

    left_order = concentrated_df["programme_display_label"].tolist()
    right_order = distributed_df["programme_display_label"].tolist()

    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=True)

    for ax, df_plot, order, title in [
        (axes[0], left_df, left_order, "Most concentrated programmes"),
        (axes[1], right_df, right_order, "Most distributed programmes"),
    ]:
        data = [
            df_plot.loc[df_plot["programme_display_label"] == label, "module_contribution_share"].dropna().to_numpy()
            for label in order
        ]
        colors = []
        for label in order:
            school_val = df_plot.loc[df_plot["programme_display_label"] == label, "school"].dropna().astype(str).iloc[0]
            colors.append(school_colors[school_val.lower()]["primary"])

        bp = ax.boxplot(data, patch_artist=True, tick_labels=order, widths=0.5)
        for patch, color in zip(bp["boxes"], colors):
            patch.set_facecolor(color)
            patch.set_edgecolor("black")
            patch.set_alpha(0.7)
        for median in bp["medians"]:
            median.set_color("black")
            median.set_linewidth(1.5)
        ax.set_title(title, fontweight="bold")
        ax.set_xticklabels(order, rotation=45, ha="right")
        ax.grid(axis="y", alpha=0.25)
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))

    axes[0].set_ylabel("Module Contribution Share")
    fig.suptitle("Result 3B: Distribution of Module Contribution Shares", fontsize=16, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()


plot_result4a_robustness_breakdown(result4_df=result4_df, top_n=5, figsize=(16, 7))
plot_result4b_module_contribution_boxplots(result4_df=result4_df, module_contrib_long_df=all_module_contrib_long_df, top_n=5, figsize=(18, 7))

most_concentrated_df, most_distributed_df = select_result4_extremes(result4_df=result4_df, top_n=5)
print("Most concentrated programmes")
display(most_concentrated_df)
print("Most distributed programmes")
display(most_distributed_df)


### Caveats and Limitations

**Purpose:** make the interpretation boundaries explicit for a final technical submission.

**What is covered:** the temporary reuse of earlier Spearman validation outputs, the runtime source used to derive 2-digit SSIC industry families, and the fact that threshold-dependent metrics are only computed after final Spearman-weighted aggregation.

**Why it closes the notebook:** the results are strongest when paired with clear limits on what the method does and does not support.


In [ ]:
def select_result4_fragility_labels(result4_df, top_n_fragile=3, top_n_robust=3):
    df = result4_df.copy()
    fragile_df = (
        df.sort_values(
            ["top1_module_share", "normalized_effective_module_count", "top5_module_share"],
            ascending=[False, True, False],
        )
        .head(top_n_fragile)
        .copy()
    )
    fragile_keys = set(zip(fragile_df["school"], fragile_df["course"]))
    remaining_df = df.loc[
        ~df.apply(lambda row: (row["school"], row["course"]) in fragile_keys, axis=1)
    ].copy()
    robust_df = (
        remaining_df.sort_values(
            ["normalized_effective_module_count", "top1_module_share", "top5_module_share"],
            ascending=[False, True, True],
        )
        .head(top_n_robust)
        .copy()
    )
    label_df = pd.concat([robust_df, fragile_df], ignore_index=True).drop_duplicates(subset=["school", "course"])
    return label_df


def plot_result4_appendix_fragility_scatter(
    result4_df,
    x_col="top1_module_share",
    y_col="normalized_effective_module_count",
    label_col="programme_display_label",
    figsize=(11, 8),
    annotate=True,
    top_n_fragile=3,
    top_n_robust=3,
    show_reference_lines=True,
):
    plot_df = result4_df.copy()
    x_vals = pd.to_numeric(plot_df[x_col], errors="coerce").astype(float)
    y_vals = pd.to_numeric(plot_df[y_col], errors="coerce").astype(float)
    x_split = float(x_vals.median())
    y_split = float(y_vals.median())

    fig, ax = plt.subplots(figsize=figsize)

    if show_reference_lines:
        ax.axvline(x=x_split, color="gray", linestyle="--", linewidth=1.1, alpha=0.8)
        ax.axhline(y=y_split, color="gray", linestyle="--", linewidth=1.1, alpha=0.8)

    for _, row in plot_df.iterrows():
        school_key = str(row["school"]).lower()
        ax.scatter(
            row[x_col],
            row[y_col],
            s=150,
            color=school_colors[school_key]["primary"],
            edgecolor="black",
            alpha=0.85,
            zorder=3,
        )

    if annotate:
        label_df = select_result4_fragility_labels(
            result4_df=plot_df,
            top_n_fragile=top_n_fragile,
            top_n_robust=top_n_robust,
        ).reset_index(drop=True)
        for i, row in label_df.iterrows():
            y_offset = 8 if i % 2 == 0 else -10
            ax.annotate(
                row[label_col],
                xy=(row[x_col], row[y_col]),
                xytext=(6, y_offset),
                textcoords="offset points",
                fontsize=8,
                bbox=dict(boxstyle="round,pad=0.22", fc="white", ec="none", alpha=0.75),
                arrowprops=dict(arrowstyle="-", color="#aaaaaa", lw=0.7, shrinkA=2, shrinkB=4),
                zorder=4,
            )

    # Quadrant labels
    x_min, x_max = ax.get_xlim()
    y_min, y_max = ax.get_ylim()

    x_left = x_min + 0.03 * (x_max - x_min)
    x_right = x_split + 0.03 * (x_max - x_min)
    y_top = y_split + 0.42 * (y_max - y_split)
    y_bottom = y_min + 0.08 * (y_split - y_min)

    quadrant_style = dict(
        fontsize=10,
        color="black",
        fontweight="bold",
        alpha=0.9,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=0.65),
    )

    ax.text(
        x_left,
        y_top,
        "Broad and Robust",
        ha="left",
        va="center",
        **quadrant_style,
    )
    ax.text(
        x_right,
        y_top,
        "Broad but with a\nDominant Module",
        ha="left",
        va="center",
        **quadrant_style,
    )
    ax.text(
        x_left,
        y_bottom,
        "Diffuse but Weakly\nSupported",
        ha="left",
        va="center",
        **quadrant_style,
    )
    ax.text(
        x_right,
        y_bottom,
        "Concentrated and Fragile",
        ha="left",
        va="center",
        **quadrant_style,
    )

    ax.set_title("Supporting Diagnostic: Contribution Concentration Map", fontweight="bold")
    ax.set_xlabel("Top 1 Module Share")
    ax.set_ylabel("Normalized Effective Module Count")
    ax.grid(alpha=0.25)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:.0%}" if x >= 0 else ""))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: f"{y:.2f}" if y >= 0 else ""))

    school_legend_handles = [
        Line2D(
            [0], [0], marker="o", color="w", label=school.upper(),
            markerfacecolor=school_colors[school]["primary"],
            markeredgecolor="black", markersize=10
        )
        for school in ["nus", "smu", "sutd"]
    ]
    ax.legend(handles=school_legend_handles, title="University", loc="upper right", frameon=True)
    plt.tight_layout()
    plt.show()


plot_result4_appendix_fragility_scatter(result4_df=result4_df)